# 🕸️ 28 · Graph neural networks and geometric deep learning
### Graphs and their matrices, graph convolutions from scratch, message passing and over-smoothing, molecules as graphs, the Weisfeiler–Lehman limit, attention on graphs, and rotation-equivariant networks for 3-D structures

**BiologyToML** · Level 2 · Research frontier · Part F — Modern machine-learning building blocks · Module 28 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/28_graph_neural_networks.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 2 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, PyTorch (CPU), scikit-learn, NetworkX (layouts only), pandas, Matplotlib |
| **Prerequisites** | Level 1 (especially Modules 03, 14, 17 and 22: matrices, eigenvectors, message passing, PyTorch); Module 26 (attention) is helpful for Section 6 but not required |
| **Estimated time** | 180 min (reading + running + the ✏️ exercises) |
| **Builds on** | [03 · Linear algebra](03_linear_algebra_for_data.ipynb), [14 · PCA](14_pca_dimensionality_reduction.ipynb), [15 · Bayesian networks](15_bayesian_networks_representation.ipynb), [17 · BN inference and learning](17_bayesian_networks_inference_learning.ipynb), [18 · Gaussian graphical models](18_gaussian_graphical_models.ipynb), [22 · Neural networks](22_neural_networks.ipynb), [23 · Model selection](23_model_selection.ipynb), [26 · Attention and transformers](26_attention_transformers.ipynb) |
| **Research frontier covered** | message-passing neural networks, graph convolutions, equivariant networks for 3-D molecules and proteins, GNNs for interaction networks and drug discovery — see [`FRONTIER_MAP.md`](../FRONTIER_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All data are **SIMULATED**, so that every prediction can be checked against a known truth:
> (1) a protein-interaction network drawn from a stochastic block model written in this notebook
> (`simulate_ppi_network`; hidden truth = the functional module of every protein);
> (2) a small-molecule library from the toolkit's `simulate_molecules` (hidden structure–activity rules and a
> noise-free `oracle` for the potency of any molecule);
> (3) idealised 3-, 4- and 5-helix protein bundles from `simulate_helix_bundle` (exact Cα coordinates).
> Real interaction networks are incomplete and biased towards well-studied proteins, real molecules are larger and
> more diverse, and real structures have side chains and flexible loops — Section 9 explains what changes.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Represent** relational biological data as graphs and **compute** the adjacency, degree, normalised adjacency and Laplacian matrices; **explain** what Laplacian eigenvectors capture and **verify** permutation equivariance and invariance numerically.
2. **Implement** a graph convolutional network (GCN) from scratch in PyTorch, **compare** it with a features-only MLP and with label propagation on a simulated protein-interaction network, and **explain** when a graph helps or hurts (homophily).
3. **Write** any GNN layer as message passing (message → aggregate → update), **relate** it to label propagation and to message passing in Bayesian networks, and **diagnose** over-smoothing in deep GNNs.
4. **Build** a message-passing network with sum pooling that predicts the potency of molecules from their graphs and **compare** it with fingerprint baselines under random and scaffold splits, against a known noise ceiling.
5. **Implement** Weisfeiler–Lehman colour refinement, **find** graphs that no message-passing GNN can tell apart, and **fix** the problem with structural features.
6. **Implement** graph attention, **check** what the attention weights learned against the true modules, and **show** that a transformer is a GNN on the complete graph.
7. **Distinguish** invariance from equivariance under rotations and translations, **compare** raw-coordinate, augmented and distance-based models on 3-D protein structures, and **implement** an E(n)-equivariant layer whose symmetry you **verify** to machine precision.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, PyTorch, course toolkit | 5 min |
| 1 · Graphs in biology and their algebra | adjacency, degree, Laplacian, eigenvectors, permutations | 20 min |
| 2 · Node classification on a protein network | MLP vs label propagation vs a GCN from scratch; homophily | 25 min |
| 3 · Message passing and over-smoothing | one template for every GNN; what depth does | 20 min |
| 4 · Graph-level prediction: molecules | atom features, an MPNN with sum pooling, fingerprints, splits | 25 min |
| 5 · What message passing cannot see | 1-WL colour refinement, indistinguishable graphs, fixes | 15 min |
| 6 · Attention on graphs | graph attention; checking attention; transformers as GNNs | 15 min |
| 7 · Geometry: invariance in 3-D | rotations, helix bundles, raw vs augmented vs distances | 20 min |
| 8 · An E(n)-equivariant layer | EGNN from scratch, numerical symmetry checks, why it matters | 20 min |
| 9 · 🧭 From toy to real data | PyG/DGL, STRING, QM9, AlphaFold, what goes wrong | 8 min |
| ✅ / 🧠 / 🚀 / 📝 | takeaways, checkpoint, discussion, going further, lab note | 7 min |

## 🧫 The biological question

Much of biology is **relational**. Proteins act in complexes and pathways, so a protein's function is written as much
in *whom it touches* as in its own sequence. A drug molecule is a set of atoms *and* the bonds between them; moving
one hydroxyl group across a ring can change its potency a hundred-fold. Cells in a tissue respond to their neighbours,
and the residues of a folded protein form a 3-D network of contacts. None of these objects is a fixed-length row of
numbers, which is what every model of Level 1 expects.

Three landmark studies show what learning directly on graphs and 3-D structures can do:

* **Antibiotic discovery.** Stokes *et al.* (2020) trained a *message-passing neural network* on roughly 2,300
  molecules screened for growth inhibition of *E. coli*, used it to rank a drug-repurposing collection and more than
  100 million molecules from the ZINC15 database, and experimentally confirmed new antibacterial compounds, among them
  **halicin**.
* **Protein function from structure.** DeepFRI (Gligorijević *et al.* 2021) runs graph convolutions over the
  residue-contact graph of a protein to predict Gene Ontology terms.
* **Protein structure.** AlphaFold 2 (Jumper *et al.* 2021) reasons about pairs of residues and builds 3-D coordinates
  with a structure module designed so that rotating or translating the whole protein does not change its predictions
  — a *symmetry* built into the architecture.

The common question: **how can a neural network learn from graphs and 3-D point clouds while respecting their
symmetries** — that relabelling the nodes, or rotating the molecule, must not change the biology? This notebook builds
the answer from pieces you already know: matrices ([Module 03](03_linear_algebra_for_data.ipynb)), eigenvectors
([Module 14](14_pca_dimensionality_reduction.ipynb)), message passing on graphs
([Module 17](17_bayesian_networks_inference_learning.ipynb)) and PyTorch training loops
([Module 22](22_neural_networks.ipynb)). The simulations mimic a STRING-like interaction network with functional
modules, a medicinal-chemistry library with hidden structure–activity rules, and designed helical-bundle proteins.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this notebook uses PyTorch (preinstalled on Colab); a CPU is enough.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn"), ("networkx", "networkx"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import time

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

In [ ]:
# ── 0.1 PyTorch configuration ──
torch.manual_seed(SEED)              # PyTorch has its own random-number generator
torch.set_num_threads(1)             # the networks here are tiny: one thread avoids threading overhead (and is polite)
T_START = time.time()                # to report the total run time at the end
print("PyTorch", torch.__version__, "| threads:", torch.get_num_threads(), "| device: cpu")

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
We use `standardize` (z-scores), `pca` and `plot_embedding` (a 2-D view of node features), `draw_graph` (small graphs with Matplotlib), the molecule simulator with its fingerprint and oracle, and the helix-bundle generator for 3-D structures. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def simulate_helix_bundle(n_helices: int = 4, helix_length: int = 14, loop_length: int = 4,
                          rng=None, noise: float = 0.0) -> dict:
    """Cα coordinates (Å) of an idealised four-helix-bundle protein, its contact map and secondary structure.

    Helices are ideal right-handed α-helices (3.6 residues per turn, 1.5 Å rise, 2.3 Å radius)
    packed antiparallel on the corners of a 10 Å square and joined by short loops.  ``noise`` adds
    Gaussian jitter (needs ``rng``).  Contacts: Cα–Cα distance < 8 Å and |i − j| ≥ 3.

    Returns a dict with ``coords`` (L × 3), ``ss`` (string of 'H'/'L'), ``distances`` and ``contacts``.
    """
    corners = 5.0 * np.array([[1, 1], [1, -1], [-1, -1], [-1, 1]], float)
    pts, ss = [], ""
    for h in range(n_helices):
        cx, cy = corners[h % 4] * (1 + h // 4)
        up = h % 2 == 0
        k = np.arange(helix_length)
        angle = np.deg2rad(100.0) * k * (1 if up else -1)      # turn with the direction of travel, so
        z = 1.5 * (k - (helix_length - 1) / 2) * (1 if up else -1)   # every helix is right-handed
        helix = np.c_[cx + 2.3 * np.cos(angle), cy + 2.3 * np.sin(angle), z]
        if pts:                                                      # loop: circular arc, 3.8 Å steps
            start, end = pts[-1], helix[0]
            d, m = np.linalg.norm(end - start), loop_length + 1
            lo, hi = 1e-6, 2 * np.pi - 1e-6                              # arc angle Θ with m equal chords
            for _ in range(60):
                theta = (lo + hi) / 2
                lo, hi = (theta, hi) if 3.8 * np.sin(theta / 2) / np.sin(theta / (2 * m)) > d else (lo, theta)
            R = 1.9 / np.sin(theta / (2 * m))
            e1 = (end - start) / d
            e2 = np.r_[0, 0, np.sign(start[2])] - np.dot(np.r_[0, 0, np.sign(start[2])], e1) * e1
            e2 /= np.linalg.norm(e2)
            c = (start + end) / 2 - R * np.cos(theta / 2) * e2
            for k in range(1, m):
                psi = -theta / 2 + k * theta / m
                pts.append(c + R * (np.sin(psi) * e1 + np.cos(psi) * e2))
            ss += "L" * loop_length
        pts.extend(helix)
        ss += "H" * helix_length
    coords = np.array(pts)
    if noise > 0:
        coords = coords + rng.normal(0, noise, coords.shape)
    D = np.linalg.norm(coords[:, None] - coords[None], axis=-1)
    sep = np.abs(np.arange(len(coords))[:, None] - np.arange(len(coords))[None])
    return dict(coords=coords, ss=ss, distances=D, contacts=(D < 8.0) & (sep >= 3))


_AR = 1.5


MOLECULE_SCAFFOLDS = {
    "benzene": ([("C", 1)] * 6, [(i, (i + 1) % 6, _AR) for i in range(6)], [0, 1, 2, 3]),
    "pyridine": ([("N", 1)] + [("C", 1)] * 5, [(i, (i + 1) % 6, _AR) for i in range(6)], [2, 3, 4]),
    "pyrimidine": ([("N", 1), ("C", 1), ("N", 1), ("C", 1), ("C", 1), ("C", 1)],
                   [(i, (i + 1) % 6, _AR) for i in range(6)], [1, 3, 4]),
    "thiophene": ([("S", 1)] + [("C", 1)] * 4, [(i, (i + 1) % 5, _AR) for i in range(5)], [1, 2, 4]),
    "furan": ([("O", 1)] + [("C", 1)] * 4, [(i, (i + 1) % 5, _AR) for i in range(5)], [1, 2, 4]),
    "cyclohexane": ([("C", 0)] * 6, [(i, (i + 1) % 6, 1) for i in range(6)], [0, 1, 2, 3]),
    "piperidine": ([("N", 0)] + [("C", 0)] * 5, [(i, (i + 1) % 6, 1) for i in range(6)], [0, 2, 3]),
    "morpholine": ([("O", 0), ("C", 0), ("C", 0), ("N", 0), ("C", 0), ("C", 0)],
                   [(i, (i + 1) % 6, 1) for i in range(6)], [3, 1, 2]),
    "naphthalene": ([("C", 1)] * 10, [(0, 1, _AR), (1, 2, _AR), (2, 3, _AR), (3, 4, _AR), (4, 5, _AR), (5, 0, _AR),
                                      (4, 6, _AR), (6, 7, _AR), (7, 8, _AR), (8, 9, _AR), (9, 5, _AR)], [0, 1, 2, 7]),
    "quinoline": ([("N", 1)] + [("C", 1)] * 9, [(0, 1, _AR), (1, 2, _AR), (2, 3, _AR), (3, 4, _AR), (4, 5, _AR),
                                                (5, 0, _AR), (4, 6, _AR), (6, 7, _AR), (7, 8, _AR), (8, 9, _AR),
                                                (9, 5, _AR)], [1, 2, 7]),
}


MOLECULE_SUBSTITUENTS = {      # attachment atom = index 0
    "methyl": ([("C", 0)], []),
    "ethyl": ([("C", 0), ("C", 0)], [(0, 1, 1)]),
    "hydroxyl": ([("O", 0)], []),
    "amino": ([("N", 0)], []),
    "fluoro": ([("F", 0)], []),
    "chloro": ([("Cl", 0)], []),
    "methoxy": ([("O", 0), ("C", 0)], [(0, 1, 1)]),
    "carboxyl": ([("C", 0), ("O", 0), ("O", 0)], [(0, 1, 2), (0, 2, 1)]),
    "amide": ([("C", 0), ("O", 0), ("N", 0)], [(0, 1, 2), (0, 2, 1)]),
    "nitrile": ([("C", 0), ("N", 0)], [(0, 1, 3)]),
    "trifluoromethyl": ([("C", 0), ("F", 0), ("F", 0), ("F", 0)], [(0, 1, 1), (0, 2, 1), (0, 3, 1)]),
    "sulfonamide": ([("S", 0), ("O", 0), ("O", 0), ("N", 0)], [(0, 1, 2), (0, 2, 2), (0, 3, 1)]),
    "dimethylamino": ([("N", 0), ("C", 0), ("C", 0)], [(0, 1, 1), (0, 2, 1)]),
    "acetyl": ([("C", 0), ("O", 0), ("C", 0)], [(0, 1, 2), (0, 2, 1)]),
    "phenyl": ([("C", 1)] * 6, [(i, (i + 1) % 6, _AR) for i in range(6)]),
}


_VALENCE = {"C": 4, "N": 3, "O": 2, "S": 2, "F": 1, "Cl": 1}


_MASS = {"C": 12.011, "N": 14.007, "O": 15.999, "S": 32.06, "F": 18.998, "Cl": 35.45, "H": 1.008}


def molecule_to_smiles(atoms, bonds) -> str:
    """Write a molecular graph as a SMILES string (depth-first, ring closures as digits; not canonical)."""
    n = len(atoms)
    nbr = {i: [] for i in range(n)}
    for i, j, o in bonds:
        nbr[i].append((j, o))
        nbr[j].append((i, o))
    for i in nbr:
        nbr[i].sort()
    parent, order, closures, seen = {0: None}, [], {}, set()

    def visit(a):                                    # first pass: tree edges vs ring-closure edges
        seen.add(a)
        order.append(a)
        for b, o in nbr[a]:
            if b not in seen:
                parent[b] = a
                visit(b)
            elif b != parent[a] and (min(a, b), max(a, b)) not in closures:
                closures[(min(a, b), max(a, b))] = o
    visit(0)
    ring_at = {i: [] for i in range(n)}
    for k, (pair, o) in enumerate(closures.items()):
        for a in pair:
            ring_at[a].append((k + 1, o, pair))

    def bond_symbol(o, a, b):
        if o == 2:
            return "="
        if o == 3:
            return "#"
        return "-" if (o == 1 and atoms[a][1] and atoms[b][1]) else ""

    def write(a):
        el, arom = atoms[a]
        s = el.lower() if arom else el
        for digit, o, pair in ring_at[a]:
            first = a == min(pair, key=order.index)
            s += (bond_symbol(o, *pair) if first else "") + (str(digit) if digit < 10 else f"%{digit}")
        kids = [(b, o) for b, o in nbr[a] if parent.get(b) == a]
        for idx, (b, o) in enumerate(kids):
            sub = bond_symbol(o, a, b) + write(b)
            s += sub if idx == len(kids) - 1 else f"({sub})"
        return s
    return write(0)


def molecule_descriptors(atoms, bonds) -> dict:
    """Simple descriptors: heavy atoms, implicit H, molecular weight, H-bond donors/acceptors,
    aromatic atoms, rings and a simplified (Crippen-like) octanol–water partition coefficient."""
    used = np.zeros(len(atoms))
    for i, j, o in bonds:
        used[i] += o
        used[j] += o
    n_h = np.array([max(0, int(np.floor(_VALENCE[el] - used[k] + 1e-9))) if el in ("C", "N", "O") else 0
                    for k, (el, _) in enumerate(atoms)])
    els = [el for el, _ in atoms]
    mw = sum(_MASS[e] for e in els) + _MASS["H"] * n_h.sum()
    hbd = sum(1 for k, e in enumerate(els) if e in ("N", "O") and n_h[k] > 0)
    hba = sum(1 for e in els if e in ("N", "O"))
    contrib = {"C": 0.25, "N": -0.75, "O": -0.55, "S": 0.4, "F": 0.35, "Cl": 0.7}
    logp = sum(contrib[e] + (0.12 if a else 0) for e, a in atoms) - 0.3 * hbd
    return dict(heavy_atoms=len(atoms), n_hydrogens=int(n_h.sum()), mol_weight=round(mw, 2), hbd=hbd, hba=hba,
                aromatic_atoms=int(sum(a for _, a in atoms)), rings=len(bonds) - len(atoms) + 1,
                logp=round(logp, 3))


def molecular_fingerprint(atoms, bonds, radius: int = 2, n_bits: int = 1024) -> np.ndarray:
    """Circular (Morgan/ECFP-like) fingerprint: hashed atom environments up to ``radius`` bonds, as 0/1 bits."""
    import zlib
    nbr = {i: [] for i in range(len(atoms))}
    for i, j, o in bonds:
        nbr[i].append((j, o))
        nbr[j].append((i, o))
    ids = [zlib.crc32(repr((el, ar, len(nbr[k]))).encode()) for k, (el, ar) in enumerate(atoms)]
    bits = np.zeros(n_bits, dtype=np.int8)
    for _ in range(radius + 1):
        for v in ids:
            bits[v % n_bits] = 1
        ids = [zlib.crc32(repr((ids[k], sorted((o, ids[j]) for j, o in nbr[k]))).encode()) for k in range(len(atoms))]
    return bits


def assemble_molecule(scaffold: str, substituents: dict):
    """Molecular graph (atoms, bonds) of ``scaffold`` carrying ``substituents`` = {site index: substituent name}."""
    atoms, bonds, sites = MOLECULE_SCAFFOLDS[scaffold]
    atoms, bonds = list(atoms), list(bonds)
    for s, name in sorted(substituents.items()):
        f_atoms, f_bonds = MOLECULE_SUBSTITUENTS[name]
        off = len(atoms)
        atoms += f_atoms
        bonds += [(i + off, j + off, o) for i, j, o in f_bonds] + [(sites[s], off, 1)]
    return atoms, bonds


def simulate_molecules(rng: np.random.Generator, n_molecules: int = 2000, noise: float = 0.3) -> dict:
    """Simulate a small-molecule library with a bioactivity (pIC50) and a solubility (logS) assay.

    Each molecule = one of 10 ring scaffolds + 1–3 substituents at the scaffold's attachment
    sites.  Hidden structure–activity rules (the "biology" a model must learn): each scaffold
    has its own baseline potency; a hydrogen-bond donor at site 0 (the key pocket) adds +1.2;
    small lipophilic groups at site 1 add +0.5; bulky groups at site 2 clash (−1.0); very
    lipophilic molecules lose potency (−0.4 per logP unit above 3); plus small random
    site × substituent effects.  Solubility falls with logP and size.

    Returns a dict with ``table`` (SMILES, scaffold, substituents, descriptors, measured
    ``pIC50`` and ``logS``), ``graphs`` (list of (atoms, bonds)), ``scaffold_effect``, and
    ``oracle(scaffold, substituents)`` → noise-free (pIC50, logS) of *any* design, e.g.
    ``oracle("pyridine", {0: "amino", 2: "methyl"})`` — the ground truth for design (Module 39).
    """
    scaffolds, subs = list(MOLECULE_SCAFFOLDS), list(MOLECULE_SUBSTITUENTS)
    weights = rng.dirichlet(np.full(len(scaffolds), 2.0))                 # some scaffolds are rare
    scaffold_effect = dict(zip(scaffolds, rng.normal(5.5, 0.8, len(scaffolds))))
    site_effect = rng.normal(0, 0.2, (4, len(subs)))
    donors = {"hydroxyl", "amino", "amide", "sulfonamide", "carboxyl"}
    lipophilic = {"methyl", "ethyl", "chloro", "trifluoromethyl"}
    bulky = {"phenyl", "sulfonamide", "dimethylamino", "trifluoromethyl"}

    def oracle(scaffold: str, substituents: dict):
        """Noise-free (pIC50, logS) of a scaffold + {site: substituent} design under the hidden rules."""
        desc = molecule_descriptors(*assemble_molecule(scaffold, substituents))
        act = scaffold_effect[scaffold] + sum(site_effect[s, subs.index(nm)] for s, nm in substituents.items())
        act += 1.2 * (substituents.get(0) in donors) + 0.5 * (substituents.get(1) in lipophilic) \
            - 1.0 * (substituents.get(2) in bulky) - 0.4 * max(0.0, desc["logp"] - 3.0)
        logs = 0.5 - 0.8 * desc["logp"] - 0.006 * (desc["mol_weight"] - 150) + 0.2 * desc["hbd"]
        return float(act), float(logs)

    rows, graphs, seen = [], [], set()
    while len(rows) < n_molecules:
        sc = scaffolds[rng.choice(len(scaffolds), p=weights)]
        n_sites = len(MOLECULE_SCAFFOLDS[sc][2])
        k = int(rng.integers(1, min(3, n_sites) + 1))
        chosen = {int(s): subs[rng.integers(len(subs))] for s in sorted(rng.choice(n_sites, k, replace=False))}
        key = (sc, tuple(sorted(chosen.items())))
        if key in seen:
            continue
        seen.add(key)
        atoms, bonds = assemble_molecule(sc, chosen)
        act, logs = oracle(sc, chosen)
        rows.append(dict(smiles=molecule_to_smiles(atoms, bonds), scaffold=sc,
                         substituents="; ".join(f"site{s}:{nm}" for s, nm in sorted(chosen.items())),
                         **molecule_descriptors(atoms, bonds),
                         pIC50=act + rng.normal(0, noise), logS=logs + rng.normal(0, noise)))
        graphs.append((atoms, bonds))
    return dict(table=pd.DataFrame(rows), graphs=graphs, scaffold_effect=scaffold_effect, oracle=oracle)


def standardize(X, mean=None, sd=None):
    """Z-score the columns of X. Pass the training mean/sd to transform test data the same way.

    Returns ``(Z, mean, sd)``; constant columns get sd = 1 so they map to 0.
    """
    X = np.asarray(X, float)
    mean = X.mean(axis=0) if mean is None else mean
    sd = X.std(axis=0) if sd is None else sd
    sd = np.where(sd > 0, sd, 1.0)
    return (X - mean) / sd, mean, sd


def pca(X, n_components: int = 2):
    """Principal component analysis by SVD of the column-centred data.

    Returns ``(scores, components, explained_variance_ratio)``: scores are N × k, components k × J.
    """
    X = np.asarray(X, float)
    Xc = X - X.mean(axis=0)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    var = S ** 2 / (len(X) - 1)
    return U[:, :n_components] * S[:n_components], Vt[:n_components], var[:n_components] / var.sum()


def draw_graph(edges, pos: dict, directed: bool = True, ax=None, highlight=None, node_color: str = "#9ecae1",
               highlight_color: str = "#fdae6b", radius: float = 0.07, title: str = "", edge_labels=None):
    """Draw a small graph with Matplotlib only.

    ``edges`` is a list of (u, v) pairs, ``pos`` maps every node to (x, y) in [0, 1]².
    ``highlight`` is a set of nodes to colour differently (e.g. observed evidence).
    """
    from matplotlib.patches import Circle, FancyArrowPatch
    ax = ax or plt.gca()
    highlight = set(highlight or [])
    for i, (u, v) in enumerate(edges):
        (x0, y0), (x1, y1) = pos[u], pos[v]
        d = np.hypot(x1 - x0, y1 - y0) or 1.0
        sx, sy = x0 + (x1 - x0) * radius / d, y0 + (y1 - y0) * radius / d
        ex, ey = x1 - (x1 - x0) * radius / d, y1 - (y1 - y0) * radius / d
        ax.add_patch(FancyArrowPatch((sx, sy), (ex, ey), arrowstyle="-|>" if directed else "-",
                                     mutation_scale=12, lw=1.2, color="0.25"))
        if edge_labels is not None:
            ax.text((x0 + x1) / 2, (y0 + y1) / 2, str(edge_labels[i]), fontsize=7, color="0.2",
                    ha="center", va="center", bbox=dict(fc="white", ec="none", alpha=0.7, pad=0.5))
    for node, (x, y) in pos.items():
        ax.add_patch(Circle((x, y), radius, fc=highlight_color if node in highlight else node_color,
                            ec="0.2", lw=1, zorder=3))
        ax.text(x, y, str(node), ha="center", va="center", fontsize=9, zorder=4)
    ax.set_xlim(-0.1, 1.1); ax.set_ylim(-0.1, 1.1); ax.set_aspect("equal"); ax.axis("off")
    if title:
        ax.set_title(title)
    return ax


def plot_embedding(Z, labels, ax=None, title: str = "", s: float = 4, legend: bool = True, cmap=None):
    """Scatter a 2-D embedding (PCA, t-SNE, latent space, …) coloured by a categorical or numeric label."""
    Z = np.asarray(Z, float)
    labels = np.asarray(labels)
    ax = ax or plt.gca()
    if labels.dtype.kind in "fc":
        sc = ax.scatter(Z[:, 0], Z[:, 1], c=labels, s=s, cmap=cmap or "viridis", linewidths=0)
        plt.colorbar(sc, ax=ax, shrink=0.8)
    else:
        palette = plt.get_cmap("tab10" if len(np.unique(labels)) <= 10 else "tab20")
        for k, lab in enumerate(np.unique(labels)):
            m = labels == lab
            ax.scatter(Z[m, 0], Z[m, 1], s=s, color=palette(k % palette.N), label=str(lab), linewidths=0)
        if legend:
            ax.legend(markerscale=3, fontsize=7, frameon=False, loc="best")
    ax.set_xticks([]); ax.set_yticks([])
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['simulate_helix_bundle', 'molecule_to_smiles', 'molecule_descriptors', 'molecular_fingerprint', 'assemble_molecule', 'simulate_molecules', 'standardize', 'pca', 'draw_graph', 'plot_embedding']))

---
## 1 · Graphs in biology and their algebra

> 📍 **Where we are.** In Part D of Level 1 graphs described *probabilistic dependencies* (Bayesian networks, Gaussian
> graphical models). Here a graph is the **data itself**: nodes carry features (expression of a protein, element of
> an atom), edges carry relations (physical interaction, chemical bond, spatial contact). Before any neural network we
> need the matrices that describe a graph, because every GNN layer is built from them.

A **graph** $G=(V,E)$ has $n$ nodes and a set of edges. We store it in the **adjacency matrix** $A$ ($n\times n$):
$A_{uv}=1$ if nodes $u$ and $v$ are linked and $0$ otherwise. Interaction networks and molecules are **undirected**,
so $A$ is symmetric. Node features are stacked in a matrix $X$ ($n\times J$, one row per node, as in Level 1).

$$d_u=\sum_v A_{uv},\qquad D=\mathrm{diag}(d_1,\dots,d_n),\qquad \hat A=\tilde D^{-1/2}(A+I)\,\tilde D^{-1/2},\qquad L=D-A .$$

*In words:* the **degree** $d_u$ counts the partners of node $u$, and $D$ puts the degrees on a diagonal. The
**normalised adjacency** $\hat A$ adds a self-loop to every node ($A+I$, with degrees $\tilde D$) and divides each
entry by $\sqrt{\tilde d_u\tilde d_v}$, so that multiplying by $\hat A$ *averages* a node's features with its
neighbours' instead of adding up ever larger sums. The **graph Laplacian** $L=D-A$ measures how much a signal changes
across edges.

🧮 **Math refresher — the Laplacian quadratic form.** For any vector $x$ with one number per node,

$$x^\top L\,x=\tfrac12\sum_{u,v}A_{uv}\,(x_u-x_v)^2 .$$

*In words:* $x^\top Lx$ adds up the squared differences across every edge. It is small when linked nodes have similar
values (a **smooth** signal on the graph) and large when neighbours disagree. Because it is a sum of squares it is never
negative, so all eigenvalues of $L$ are $\ge 0$; the constant vector has $x^\top Lx=0$, so the smallest eigenvalue is
$0$. The eigenvectors with small eigenvalues are the *smoothest* patterns the graph allows — exactly as the first
principal components of [Module 14](14_pca_dimensionality_reduction.ipynb) are the directions of largest variance.

In [ ]:
# ── 1.1 A toy interaction graph and its matrices ──
toy_nodes = ["A", "B", "C", "D", "E", "F"]
toy_edges = [("A", "B"), ("A", "C"), ("B", "C"), ("C", "D"), ("D", "E"), ("D", "F")]   # a triangle and a star
idx = {v: i for i, v in enumerate(toy_nodes)}
A_toy = np.zeros((6, 6))
for u, v in toy_edges:
    A_toy[idx[u], idx[v]] = A_toy[idx[v], idx[u]] = 1          # undirected → symmetric


def normalized_adjacency(A):
    """GCN propagation matrix  D̃^(-1/2) (A + I) D̃^(-1/2)  (Kipf & Welling 2017)."""
    A_tilde = A + np.eye(len(A))
    d = A_tilde.sum(axis=1)
    return A_tilde / np.sqrt(d[:, None] * d[None, :])


deg = A_toy.sum(axis=1)
L_toy = np.diag(deg) - A_toy
A_hat_toy = normalized_adjacency(A_toy)
print("degrees:", dict(zip(toy_nodes, deg.astype(int))))
print("Laplacian L = D − A:\n", L_toy.astype(int))
print("row sums of L (always 0):", L_toy.sum(axis=1))

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
pos = {"A": (0.05, 0.8), "B": (0.05, 0.2), "C": (0.35, 0.5), "D": (0.65, 0.5), "E": (0.95, 0.8), "F": (0.95, 0.2)}
draw_graph(toy_edges, pos, directed=False, ax=axes[0], title="toy interaction graph")
for ax, M, name in zip(axes[1:], [A_toy, A_hat_toy, L_toy], ["adjacency A", "normalised Â", "Laplacian L = D − A"]):
    im = ax.imshow(M, cmap="RdBu_r", vmin=-np.abs(M).max(), vmax=np.abs(M).max())
    ax.set_xticks(range(6)); ax.set_xticklabels(toy_nodes); ax.set_yticks(range(6)); ax.set_yticklabels(toy_nodes)
    ax.set_title(name); plt.colorbar(im, ax=ax, shrink=0.75)
plt.tight_layout(); plt.show()

In [ ]:
# ── 1.2 Check the Laplacian facts numerically ──
x = np.array([1.0, 1.2, 0.9, -0.5, -1.0, -0.8])          # a signal: high on the triangle, low on the star
quad = x @ L_toy @ x
edge_sum = 0.5 * sum(A_toy[u, v] * (x[u] - x[v]) ** 2 for u in range(6) for v in range(6))
print(f"x^T L x = {quad:.4f}   ½ Σ A_uv (x_u − x_v)^2 = {edge_sum:.4f}")
eigval, eigvec = np.linalg.eigh(L_toy)                    # symmetric matrix → real eigenvalues, orthogonal eigenvectors
print("eigenvalues of L:", eigval.round(3))
print("eigenvector of the 0 eigenvalue (constant):", eigvec[:, 0].round(3))
print("second eigenvector (Fiedler vector):     ", dict(zip(toy_nodes, eigvec[:, 1].round(2))))

🔍 **What just happened?** The quadratic form equals the sum over edges, as the refresher promised. All eigenvalues are
$\ge0$, the smallest is $0$ with a constant eigenvector (a graph with one connected piece has exactly one zero
eigenvalue — try deleting the edge C–D). The **second** eigenvector, called the **Fiedler vector**, is the smoothest
non-constant pattern: it takes one sign on the triangle {A, B, C} and the other on the star {D, E, F}, cutting the graph
through its weakest link. That is the principle of **spectral clustering**, and it suggests that eigenvectors of graph
matrices can reveal modules — let us check on a bigger graph where we *know* the modules.

In [ ]:
# ── 1.3 🧪 Ground-truth check: does the Fiedler vector find two hidden modules? ──
rng = np.random.default_rng(SEED + 1)
modules = np.repeat([0, 1], 20)                             # 40 proteins in two modules (SIMULATED)
p_edge = np.where(modules[:, None] == modules[None, :], 0.30, 0.03)
A_two = np.triu(rng.random((40, 40)) < p_edge, 1).astype(float)
A_two = A_two + A_two.T
val2, vec2 = np.linalg.eigh(np.diag(A_two.sum(1)) - A_two)
fiedler = vec2[:, 1]
split = (fiedler > 0).astype(int)
agree = max(np.mean(split == modules), np.mean(split != modules))   # the sign of an eigenvector is arbitrary
print(f"edges within modules: {int(np.sum(A_two * (modules[:, None] == modules)) / 2)}, "
      f"between: {int(np.sum(A_two * (modules[:, None] != modules)) / 2)}")
print(f"sign of the Fiedler vector recovers the true module of {agree:.0%} of the proteins")

fig, ax = plt.subplots(figsize=(6.5, 2.8))
ax.bar(range(40), fiedler, color=np.where(modules == 0, "tab:blue", "tab:orange"))
ax.axhline(0, color="k", lw=0.6)
ax.set_xlabel("protein (ordered by true module: blue = module 0, orange = module 1)")
ax.set_ylabel("Fiedler vector entry")
ax.set_title("SIMULATED two-module network: the 2nd Laplacian eigenvector separates the modules")
plt.tight_layout(); plt.show()

🔍 **What just happened?** With many more edges inside modules than between them, the Fiedler vector is positive on one
module and negative on the other, and its sign alone recovers the hidden module of every protein (or nearly every one
— the printed number is the exact score). Eigenvectors of graph matrices are the graph's "Fourier modes": low
eigenvalues = smooth, large-scale patterns. We will meet them again as **positional encodings** for graph
transformers (Section 6 and 🚀 Bonus) and, implicitly, in the smoothing done by every GCN layer (Section 3).

> 🔗 **Builds on** [Module 14 · PCA](14_pca_dimensionality_reduction.ipynb), Section on eigen-decomposition: PCA keeps
> the eigenvectors of the covariance matrix with the *largest* eigenvalues; spectral methods keep the eigenvectors of
> the Laplacian with the *smallest* eigenvalues. Both are "find the most important directions of a symmetric matrix".

### Permutation symmetry: the rule every graph model must obey

The numbering of the nodes is arbitrary: protein 17 could have been protein 3. If we reorder the nodes with a
**permutation matrix** $P$ (an identity matrix with its rows shuffled), the features become $PX$ and the adjacency
$PAP^\top$. A layer $f$ that computes one output per node must be **permutation-equivariant** and a whole-graph
prediction $g$ must be **permutation-invariant**:

$$f(PX,\;PAP^\top)=P\,f(X,A),\qquad g(PX,\;PAP^\top)=g(X,A).$$

*In words:* relabelling the nodes must simply relabel the node outputs (equivariance) and must leave a graph-level
answer unchanged (invariance). An MLP applied to the flattened adjacency matrix breaks this rule; a GCN layer obeys it
automatically, because it uses the *same* weights for every node and only sums over neighbours.

In [ ]:
# ── 1.4 Permutation equivariance (GCN layer) vs no symmetry (MLP on the flattened adjacency) ──
rng = np.random.default_rng(SEED + 2)
X_toy = rng.normal(size=(6, 3))                           # 3 features per node
W_toy = rng.normal(size=(3, 2))                           # ONE weight matrix shared by all nodes


def gcn_layer_np(A, X, W):
    """One graph-convolution layer in NumPy: ReLU(Â X W)."""
    return np.maximum(0, normalized_adjacency(A) @ X @ W)


P = np.eye(6)[rng.permutation(6)]                          # a random relabelling of the nodes
out, out_perm = gcn_layer_np(A_toy, X_toy, W_toy), gcn_layer_np(P @ A_toy @ P.T, P @ X_toy, W_toy)
print("GCN: f(PX, PAPᵀ) == P f(X, A)?        ", np.allclose(out_perm, P @ out))
print("sum readout invariant?                 ", np.allclose(out_perm.sum(0), out.sum(0)))

W_mlp = rng.normal(size=36)                                # an MLP-style model that reads the 36 adjacency entries
print(f"MLP on flattened A: output {float(A_toy.ravel() @ W_mlp):+.3f} before, "
      f"{float((P @ A_toy @ P.T).ravel() @ W_mlp):+.3f} after relabelling (same graph!)")

🔍 **What just happened?** The GCN layer passed both symmetry tests to machine precision, while a model that reads the
adjacency matrix as a flat vector gave two different answers for the *same* graph written in two node orders. With $n$
nodes there are $n!$ orders ($6!=720$ here, astronomically many for a 600-protein network), so "learning the symmetry
from data" is hopeless; **building it into the architecture** is the central idea of graph neural networks and, in
Sections 7–8, of geometric deep learning.

⚠️ **Undirected vs directed.** Regulatory and signalling graphs are directed (TF → target, kinase → substrate); then $A$
is not symmetric and one usually passes messages along and against the arrows with separate weights. Everything below
assumes undirected graphs.

✏️ **Your turn.** Delete the edge ("C", "D") from `toy_edges` and re-run 1.1–1.2. How many eigenvalues of $L$ are now
zero, and what do the corresponding eigenvectors look like?

---
## 2 · Node classification on a protein-interaction network

> 📍 **Where we are.** We have the matrices; now a real task. Many proteins in any organism have no functional
> annotation. Interaction databases such as STRING and BioGRID link them to annotated partners, and expression
> profiles give each protein noisy features of its own. **Semi-supervised node classification** predicts the unknown
> labels from the few known ones, the features and the graph.

**The simulated data.** `simulate_ppi_network` draws a **stochastic block model** (Holland, Laskey & Leinhardt 1983):
600 proteins in 4 functional modules (e.g. a kinase cascade, the proteasome, a splicing complex, a metabolic pathway);
two proteins in the *same* module interact with probability $p_{\text{in}}$, two in *different* modules with
$p_{\text{out}}$. Each protein also has a 50-dimensional, expression-like feature vector: its module's mean profile
plus a lot of noise. Only **10 % of the proteins are labelled**. The hidden truth is the module of every protein.

The fraction of edges that join two proteins of the same module is called the **homophily** $h$ of the graph. Real
interaction networks are strongly but not perfectly homophilous, and that number decides whether a graph helps.

🔬 **Assay box — how a protein-interaction network is measured.** The edges in STRING or BioGRID come from assays, each
with its own bias. **Yeast two-hybrid (Y2H)** fuses one protein to a DNA-binding domain and another to an activation
domain and reports a direct binary interaction when the pair reconstitutes a transcription factor — it finds direct,
often transient binary contacts but misses interactions needing a third partner and gives false positives from
self-activators. **Affinity purification–mass spectrometry (AP-MS)** pulls down a tagged "bait" protein with everything
stuck to it and identifies the "prey" by mass spectrometry — it recovers stable complexes (co-complex, not necessarily
direct) and is biased towards abundant, easily-tagged proteins. Both share a deeper bias: **well-studied proteins are
tested far more often**, so they accumulate more recorded partners. A GNN can learn this "study bias" (degree ≈ fame)
instead of real function — the pitfall we return to in the 🧭 section. Our simulation has none of these artefacts; it
gives a clean stochastic-block-model graph so we can isolate *when a graph helps at all*.

In [ ]:
# ── 2.1 A SIMULATED protein-interaction network with hidden functional modules ──
def simulate_ppi_network(rng, n=600, n_modules=4, p_in=0.04, p_out=0.005, feature_noise=3.0, n_features=50):
    """Stochastic block model + noisy module-specific features.

    Returns a dict with the adjacency ``A`` (n × n, symmetric 0/1), features ``X`` (n × n_features),
    the true module ``y`` of every node and the module mean ``profiles``.
    """
    y = rng.permutation(np.repeat(np.arange(n_modules), n // n_modules))
    p = np.where(y[:, None] == y[None, :], p_in, p_out)            # edge probability for every pair
    A = np.triu(rng.random((n, n)) < p, 1)
    A = (A | A.T).astype(float)
    profiles = rng.normal(0, 1, (n_modules, n_features))
    X = profiles[y] + rng.normal(0, feature_noise, (n, n_features))
    return dict(A=A, X=X, y=y, profiles=profiles)


def homophily(A, y):
    """Fraction of edges whose two end points share a label."""
    return float(np.sum(A * (y[:, None] == y[None, :])) / np.sum(A))


rng = np.random.default_rng(SEED)
ppi = simulate_ppi_network(rng)
A_ppi, y_ppi = ppi["A"], ppi["y"]
X_ppi = standardize(ppi["X"])[0]
labelled = np.sort(rng.choice(600, 60, replace=False))           # 10 % of the proteins have a known module
unlabelled = np.setdiff1d(np.arange(600), labelled)
print("SIMULATED protein-interaction network")
print(f"  proteins: {len(y_ppi)}, interactions: {int(A_ppi.sum() / 2)}, mean degree: {A_ppi.sum(1).mean():.2f}")
print(f"  homophily h = {homophily(A_ppi, y_ppi):.3f}   isolated proteins: {int(np.sum(A_ppi.sum(1) == 0))}")
print(f"  labelled proteins: {len(labelled)} ({np.bincount(y_ppi[labelled]).tolist()} per module)")

In [ ]:
# ── 2.2 Look at the data: the graph has block structure, the features overlap ──
order = np.argsort(y_ppi, kind="stable")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
a1.spy(A_ppi[np.ix_(order, order)], markersize=0.6, color="k")
a1.set_title("adjacency, proteins sorted by true module\n(SIMULATED: dense blocks = modules)")
a1.set_xlabel("protein"); a1.set_ylabel("protein")
scores, _, evr = pca(X_ppi, 2)
plot_embedding(scores, np.array([f"module {m}" for m in y_ppi]), ax=a2, s=8,
               title=f"features only: PCA of the expression-like profiles\n(PC1+PC2 explain {evr.sum():.0%} of the variance)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Sorted by their (hidden) module, the proteins' interactions form four dense diagonal blocks
plus a sprinkling of cross-module edges; with a homophily of about 0.73, roughly three edges in four connect proteins of
the same module. The features, in contrast, overlap heavily in a PCA view: each protein's profile is mostly noise.
Neither source alone is clean. Let us try three classifiers:

1. **MLP on the features** (ignores the graph) — the Level 1 approach.
2. **Label spreading** (ignores the features) — repeatedly let every node average its neighbours' label
   distributions while being pulled back towards its own known label (Zhou *et al.* 2004). This is the soft "local and
   global consistency" variant; the closely related **label propagation** of Zhu & Ghahramani (2002) instead *clamps*
   the known labels to stay fixed. Our code (next cell) implements the spreading form $F\leftarrow\alpha SF+(1-\alpha)Y_0$.
3. **A graph convolutional network** (uses both). Kipf & Welling (2017) define a layer as

$$H^{(\ell+1)}=\sigma\!\left(\hat A\,H^{(\ell)}\,W^{(\ell)}\right),\qquad H^{(0)}=X .$$

*In words:* every layer first transforms each node's current representation with the *same* weight matrix
$W^{(\ell)}$ (a linear layer, as in [Module 22](22_neural_networks.ipynb)), then replaces it by the degree-normalised
average over itself and its neighbours ($\hat A$), then applies a nonlinearity $\sigma$ (ReLU). Two layers mean that a
protein's prediction depends on everything within two interaction steps. The last layer has 4 outputs, turned into
class probabilities by a softmax and trained with cross-entropy on the 60 labelled proteins only.

A neat consequence: **an MLP is a GCN on a graph with no edges** ($\hat A=I$). So one class does both.

In [ ]:
# ── 2.3 A GCN from scratch (dense matrices) and its training loop ──
class GCN(nn.Module):
    """Stack of graph convolutions H ← ReLU(Â H W); with Â = I it is an ordinary MLP."""

    def __init__(self, d_in, d_hidden, d_out, n_layers=2, dropout=0.5):
        super().__init__()
        dims = [d_in] + [d_hidden] * (n_layers - 1) + [d_out]
        self.linears = nn.ModuleList(nn.Linear(a, b) for a, b in zip(dims[:-1], dims[1:]))
        self.dropout = dropout

    def forward(self, A_hat, X):
        H = X
        for k, lin in enumerate(self.linears):
            H = F.dropout(H, self.dropout, self.training)      # randomly silence inputs while training (regularises)
            H = A_hat @ lin(H)                                 # transform with shared W, then average over neighbours
            if k < len(self.linears) - 1:
                H = F.relu(H)
        return H                                               # logits, one row per node


def train_node_classifier(A_hat, X, y, train_idx, n_layers=2, epochs=200, seed=SEED):
    """Full-batch training on the labelled nodes; returns the predicted label of every node."""
    torch.manual_seed(seed)
    A_t, X_t, y_t = (torch.tensor(A_hat, dtype=torch.float32), torch.tensor(X, dtype=torch.float32),
                     torch.tensor(y))
    model = GCN(X.shape[1], 16, int(y.max()) + 1, n_layers)
    opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    tr = torch.tensor(train_idx)
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        loss = F.cross_entropy(model(A_t, X_t)[tr], y_t[tr])   # the loss sees ONLY the labelled nodes
        loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        return model(A_t, X_t).argmax(1).numpy()


print(GCN(50, 16, 4))
print("parameters:", sum(p.numel() for p in GCN(50, 16, 4).parameters()), "= 50·16 + 16 + 16·4 + 4")

In [ ]:
# ── 2.4 Label spreading: message passing with no parameters ──
def label_propagation(A, y, train_idx, n_classes, alpha=0.9, n_iter=50):
    """Label spreading (Zhou et al. 2004): F ← α S F + (1 − α) Y0, S the normalised adjacency (no self-loops).

    This is the *soft* variant — a node is pulled back towards its own label but never clamped to it;
    the Zhu & Ghahramani (2002) label-propagation variant instead holds the known labels fixed."""
    d = A.sum(1)
    d[d == 0] = 1                                              # isolated nodes: avoid dividing by zero
    S = A / np.sqrt(d[:, None] * d[None, :])
    Y0 = np.zeros((len(A), n_classes))
    Y0[train_idx, y[train_idx]] = 1                            # one-hot rows for the labelled nodes, zeros elsewhere
    F_lab = Y0.copy()
    for _ in range(n_iter):
        F_lab = alpha * S @ F_lab + (1 - alpha) * Y0           # average the neighbours, pull back towards the labels
    return F_lab.argmax(1)


t0 = time.time()
pred = {"MLP (features only)": train_node_classifier(np.eye(600), X_ppi, y_ppi, labelled),
        "label propagation (graph only)": label_propagation(A_ppi, y_ppi, labelled, 4),
        "GCN (features + graph)": train_node_classifier(normalized_adjacency(A_ppi), X_ppi, y_ppi, labelled)}
print("🧪 Ground-truth check — accuracy on the 540 UNLABELLED proteins (true modules known):")
for name, p in pred.items():
    print(f"   {name:32s} {np.mean(p[unlabelled] == y_ppi[unlabelled]):.3f}")
print(f"(three models trained in {time.time() - t0:.1f} s)")

🔍 **What just happened?** The features alone (MLP) recover roughly two thirds of the modules and the graph alone (label
propagation) about the same, but the GCN, which mixes each protein's noisy profile with those of its partners,
classifies about 97 % of the unlabelled proteins correctly — with the same 60 labels. Averaging over neighbours is
**denoising**: if most of your partners share your module, their average profile is a much cleaner estimate of your
module's profile than your own measurement.

⚠️ **Transductive vs inductive.** The GCN saw the features and edges of *all* 600 proteins during training (only the
labels were hidden). That is fine for annotating a fixed network ("transductive"), but it is not the same as predicting
for a new protein added later ("inductive", e.g. GraphSAGE). When you read a paper, check which setting it reports.

But is a graph always helpful? The denoising argument assumed that neighbours tend to share labels. Let us vary the
homophily $h$ while keeping the mean degree at 8.

In [ ]:
# ── 2.5 When does the graph help? Vary homophily at a fixed mean degree ──
t0 = time.time()
h_grid = [0.05, 0.15, 0.25, 0.4, 0.55, 0.7, 0.85]
results = {"MLP (features only)": [], "label propagation": [], "GCN": []}
for s in range(2):                                             # two simulated networks per setting
    r = np.random.default_rng(SEED + 100 + s)
    for k, h in enumerate(h_grid):
        net = simulate_ppi_network(np.random.default_rng(SEED + 100 + s), p_in=h * 8 / 149, p_out=(1 - h) * 8 / 450)
        Xs, yh = standardize(net["X"])[0], net["y"]
        tr = np.sort(r.choice(600, 60, replace=False)) if k == 0 else tr
        te = np.setdiff1d(np.arange(600), tr)
        if k == 0:                                             # the features do not depend on h: fit the MLP once
            mlp_acc = np.mean(train_node_classifier(np.eye(600), Xs, yh, tr)[te] == yh[te])
        results["MLP (features only)"].append(mlp_acc)
        results["label propagation"].append(np.mean(label_propagation(net["A"], yh, tr, 4)[te] == yh[te]))
        results["GCN"].append(np.mean(train_node_classifier(normalized_adjacency(net["A"]), Xs, yh, tr)[te] == yh[te]))
sweep = pd.DataFrame({k: np.mean(np.reshape(v, (2, -1)), axis=0) for k, v in results.items()}, index=h_grid)
sweep.index.name = "homophily h"
print(sweep.round(3)); print(f"({time.time() - t0:.0f} s)")

fig, ax = plt.subplots(figsize=(6.5, 3.4))
for (name, col), c in zip(sweep.items(), ["tab:gray", "tab:green", "tab:blue"]):
    ax.plot(sweep.index, col, "o-", color=c, label=name)
ax.axvline(0.25, color="k", ls=":", lw=0.8); ax.text(0.26, 0.2, "h = 1/4: edges ignore modules", fontsize=7)
ax.set_xlabel("homophily h (fraction of edges within a module)"); ax.set_ylabel("accuracy on unlabelled proteins")
ax.set_title("SIMULATED: a GCN helps only when neighbours tend to share labels"); ax.legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** The MLP is a flat line — it never looks at the edges. Label propagation tracks the homophily
almost linearly. The GCN beats the features-only model by a wide margin once $h$ exceeds about one half, but **below it
the GCN is *worse* than ignoring the graph**: at $h=0.25$ the edges carry no information about modules, and at
$h<0.25$ (heterophily: partners tend to be in *other* modules) averaging actively mixes in the wrong profiles.

⚠️ **Biological heterophily is real.** Many interactions link *different* kinds of entities: kinases and their
substrates, ligands and receptors, a transcription factor and the genes it regulates in other pathways. Always ask
whether "neighbours are similar" holds for your labels before choosing a smoothing GNN; there are architectures that
treat self- and neighbour-information separately (GraphSAGE's concatenation, attention in Section 6).

✏️ **Your turn.** In 2.4, change the number of labelled proteins from 60 to 12 (2 %) and to 300 (50 %). For which method
does the number of labels matter most, and why?

---
## 3 · Message passing: one template for every GNN, and what depth does

> 📍 **Where we are.** The GCN and label propagation both repeat "combine each node with its neighbours". Gilmer *et
> al.* (2017) showed that almost every GNN fits one template, **message passing**, and that view also explains why deep
> GNNs fail.

In each layer, every node $v$ receives messages from its neighbours $\mathcal N(v)$, aggregates them with an operation
that does not care about their order (sum, mean, max), and updates its state:

$$m_v=\bigoplus_{u\in\mathcal N(v)}\;\phi_{\text{msg}}\!\left(h_u,h_v,e_{uv}\right),\qquad h_v'=\phi_{\text{upd}}\!\left(h_v,m_v\right).$$

*In words:* $\phi_{\text{msg}}$ computes what neighbour $u$ tells $v$ (it may depend on both nodes and on the edge's
features $e_{uv}$, such as a bond type); $\bigoplus$ pools the messages into one vector; $\phi_{\text{upd}}$ combines
the pooled message with $v$'s own state. Because $\bigoplus$ is order-independent and the same functions are used at
every node, the layer is automatically permutation-equivariant (Section 1). The choices define the architectures:

| Model | message | aggregate | update |
|---|---|---|---|
| Label spreading | neighbour's label distribution | normalised sum | pull back towards the labels (no parameters) |
| GCN (Kipf & Welling 2017) | $W h_u/\sqrt{\tilde d_u\tilde d_v}$ | sum (incl. self-loop) | ReLU |
| GraphSAGE (Hamilton *et al.* 2017) | $h_u$ | mean / max | ReLU of $W[h_v \Vert m_v]$ (self kept separate) |
| GIN (Xu *et al.* 2019) | $h_u$ | sum | MLP of $(1+\epsilon)h_v+m_v$ |
| GAT (Veličković *et al.* 2018) | $\alpha_{vu}W h_u$ (learned attention) | sum | nonlinearity |
| MPNN (Gilmer *et al.* 2017) | neural network of $(h_u, h_v, e_{uv})$ | sum | gated recurrent unit |

> 🔗 **Builds on** [Module 17 · Inference](17_bayesian_networks_inference_learning.ipynb), Section 3: on a polytree,
> variable elimination is equivalent to a **message-passing** algorithm (belief propagation), in which each node sends
> its neighbour a small table summarising everything on its side of the graph,
> $m_{u\to v}(x_v)=\sum_{x_u}P(x_v\mid x_u)\,\phi_u(x_u)\prod_{w\in\mathcal N(u)\setminus v}m_{w\to u}(x_u)$. A GNN keeps
> the *pattern* — local messages, combined at each node, iterated — but replaces the hand-derived probability tables by
> learned vectors and learned functions. Belief propagation is exact on trees; a GNN makes no such promise, it is simply
> trained end-to-end to make the final prediction good.

In [ ]:
# ── 3.1 A generic message-passing step, and two special cases ──
def message_passing_step(H, A, message, aggregate, update):
    """One round of message passing on a dense graph. message(h_u, h_v) → vector; aggregate(list) → vector."""
    out = []
    for v in range(len(H)):
        msgs = [message(H[u], H[v], u, v) for u in np.flatnonzero(A[v])]
        out.append(update(H[v], aggregate(msgs) if msgs else np.zeros_like(H[v])))
    return np.array(out)


# Special case 1: a GCN layer (messages include the self-loop, so we pass A + I)
A_hat_n = normalized_adjacency(A_toy)
gcn_mp = message_passing_step(X_toy @ W_toy, A_toy + np.eye(6),
                              message=lambda hu, hv, u, v: A_hat_n[v, u] * hu,
                              aggregate=lambda m: np.sum(m, axis=0), update=lambda hv, m: np.maximum(0, m))
print("generic message passing == GCN layer of 1.4:", np.allclose(gcn_mp, gcn_layer_np(A_toy, X_toy, W_toy)))

# Special case 2: one label-propagation step (alpha = 1, no clamping)
Y_toy = np.eye(2)[[0, 0, 0, 1, 1, 1]] * np.array([[1], [0], [0], [1], [0], [0]])   # only A and D labelled
d_t = A_toy.sum(1)
lp_mp = message_passing_step(Y_toy, A_toy, message=lambda hu, hv, u, v: hu / np.sqrt(d_t[u] * d_t[v]),
                             aggregate=lambda m: np.sum(m, axis=0), update=lambda hv, m: m)
S_toy = A_toy / np.sqrt(d_t[:, None] * d_t[None, :])
print("generic message passing == label propagation step:", np.allclose(lp_mp, S_toy @ Y_toy))

🔍 **What just happened?** One loop-based function reproduced both the GCN layer of Section 1 and a step of label
propagation, just by plugging in different message, aggregate and update functions. Production libraries do exactly
this, but vectorised over millions of edges with sparse "scatter-add" operations instead of a Python loop.

### Over-smoothing: why GNNs are shallow

A $K$-layer GNN sees the $K$-hop neighbourhood of every node, so deeper should mean "more context". But each layer also
**averages**. Without the weights and nonlinearities, $K$ GCN layers compute $\hat A^K X$, and as $K$ grows every row
converges to the same direction (the leading eigenvector of $\hat A$, which depends only on node degrees): the nodes
become indistinguishable. Li, Han & Wu (2018) named this **over-smoothing**. We measure it with the normalised
**Dirichlet energy**

$$E(H)=\frac{\operatorname{tr}\!\left(H^\top(I-\hat A)\,H\right)}{\operatorname{tr}(H^\top H)},$$

*In words:* the Laplacian quadratic form of Section 1 (with the normalised Laplacian $I-\hat A$), summed over feature
columns and divided by the total size of the features — how much neighbouring rows still differ, relative to how big
they are. $E\to0$ means "all nodes look alike". Next to it we measure what matters for biology: the accuracy of a
logistic regression ([Module 09](09_logistic_regression.ipynb)) trained on the 60 labelled proteins using the smoothed
features $\hat A^K X$. (This "smooth, then fit a linear model" recipe is a real method, SGC: Wu *et al.* 2019.)

In [ ]:
# ── 3.2 Over-smoothing without any learning: propagate the features K times ──
A_hat_ppi = normalized_adjacency(A_ppi)
L_norm = np.eye(600) - A_hat_ppi
K_grid = [0, 1, 2, 4, 8, 16, 32, 64, 128]
energy, lr_acc, H, k_done = [], [], X_ppi.copy(), 0
for K in K_grid:
    while k_done < K:                                          # H = Â^K X, computed incrementally
        H, k_done = A_hat_ppi @ H, k_done + 1
    energy.append(np.trace(H.T @ L_norm @ H) / np.trace(H.T @ H))
    Hs = H / H.std(0)                                          # rescale: smoothing also shrinks the numbers
    clf = LogisticRegression(C=1.0, max_iter=2000).fit(Hs[labelled], y_ppi[labelled])
    lr_acc.append(np.mean(clf.predict(Hs[unlabelled]) == y_ppi[unlabelled]))
print(pd.DataFrame({"K": K_grid, "Dirichlet energy": np.round(energy, 4), "accuracy": np.round(lr_acc, 3)}).to_string(index=False))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.3))
Kp = np.array(K_grid) + 1                                      # plot K + 1 on a log axis so that K = 0 fits
a1.semilogx(Kp, energy, "o-", color="tab:purple"); a1.set_xlabel("K + 1  (K = propagation steps)")
a1.set_ylabel("normalised Dirichlet energy"); a1.set_title("neighbours become identical as K grows")
a2.semilogx(Kp, lr_acc, "o-", color="tab:blue"); a2.set_xlabel("K + 1"); a2.set_ylabel("accuracy (unlabelled)")
a2.set_title("SIMULATED network: a little smoothing denoises,\ntoo much erases the modules")
plt.tight_layout(); plt.show()

In [ ]:
# ── 3.3 Over-smoothing in trained GCNs: accuracy vs number of layers ──
t0 = time.time()
depth_acc = {L: np.mean([np.mean(train_node_classifier(A_hat_ppi, X_ppi, y_ppi, labelled, n_layers=L, seed=s)[unlabelled]
                                 == y_ppi[unlabelled]) for s in range(2)]) for L in [1, 2, 4, 6, 8]}
print("trained GCN, mean accuracy over 2 seeds:")
for L, a in depth_acc.items():
    print(f"   {L} layer{'s' if L > 1 else ' '}: {a:.3f}")
print(f"({time.time() - t0:.0f} s)")

🔍 **What just happened?** Without any learning, the Dirichlet energy falls by orders of magnitude as $K$ grows. The
accuracy first *rises* — a few rounds of averaging denoise the profiles, as in Section 2 — peaks at a handful of steps,
then holds for a surprisingly long time before collapsing towards chance (0.25) only at $K\gtrsim64$. Note the apparent
paradox: the energy is already ~$10^{-4}$ at $K=32$ yet accuracy is still ~0.94. The reason is that we **rescale** the
smoothed features (`H / H.std(0)`) before the classifier, which blows the vanishingly small *residual* differences
between nodes back up to unit scale — so near-zero energy does not yet mean the information is gone, it means it survives
only in ever-tinier differences that are increasingly fragile to noise and finite precision. By $K=64$ those differences
have shrunk below what rescaling can recover and the accuracy drops. Trained GCNs over-smooth sooner: one to four layers
work well, while 6- and especially 8-layer GCNs lose accuracy (deep GCNs are also harder to optimise — the signal is
averaged *and* passed through many weight matrices, with no rescaling to rescue it).

This is why most GNNs in biology use **2–5 layers**, and why tricks such as residual connections ($h' = h + f(h)$),
normalisation, and attention (which can choose *not* to average) were invented. A protein's function can depend on
partners far away; capturing that without over-smoothing is an active research topic (it is related to
"over-squashing": too much information squeezed through a few bottleneck edges).

✏️ **Your turn.** Add a residual connection to the `GCN.forward` loop (keep `H` and add the new value when the shapes
match, i.e. for the hidden layers) and re-run 3.3. Do the 6- and 8-layer networks recover?

---
## 4 · Graph-level prediction: molecules as graphs

> 📍 **Where we are.** So far each *node* had a label. In drug discovery each *graph* — a molecule — has a label:
> potency against a target, solubility, toxicity. We need one more ingredient, a **readout** that turns node
> representations into one vector per graph, and we must respect the second symmetry rule of Section 1 (invariance).

**The simulated library.** `simulate_molecules` builds molecules from 10 ring scaffolds (benzene, pyridine, thiophene,
…) and 15 substituents (hydroxyl, amino, chloro, phenyl, …) attached at defined sites. Its hidden structure–activity
rules include a scaffold-specific baseline potency, a hydrogen-bond donor at site 0 (+1.2 pIC50 units), a small
lipophilic group at site 1 (+0.5), a bulky group at site 2 (−1.0) and a penalty for very lipophilic molecules; the
measured pIC50 adds noise (sd 0.3). The simulator also returns a noise-free **oracle**, so we know how well a perfect
model could do. (pIC50 $=-\log_{10}$ IC50 in mol/L: a pIC50 of 6 means half-maximal inhibition at 1 µM; one unit = ten-fold.
[Module 40](40_molecular_ml_drug_discovery.ipynb) is devoted to this library and to drug-discovery practice.)

In [ ]:
# ── 4.1 A SIMULATED medicinal-chemistry library ──
rng = np.random.default_rng(SEED + 4)
lib = simulate_molecules(rng, 2000)
mols = lib["table"]
graphs = lib["graphs"]                                        # list of (atoms, bonds)
print("SIMULATED library:", len(mols), "molecules,", mols.scaffold.nunique(), "scaffolds")
print(mols[["smiles", "scaffold", "substituents", "heavy_atoms", "pIC50"]].head(6).to_string())
atoms0, bonds0 = graphs[0]
print("\nthe first molecule as a graph:\n  atoms (element, aromatic):", atoms0, "\n  bonds (i, j, order):", bonds0)

In [ ]:
# ── 4.2 Draw molecular graphs with Matplotlib (positions from a force-directed layout) ──
ELEMENT_COLOURS = {"C": "#bdbdbd", "N": "#6baed6", "O": "#fb6a4a", "S": "#fdd835", "F": "#74c476", "Cl": "#31a354"}


def draw_molecule(atoms, bonds, ax, title=""):
    """Atoms as coloured discs; single bonds one line, double two, triple three, aromatic solid + dotted."""
    G = nx.Graph()
    G.add_nodes_from(range(len(atoms)))
    G.add_edges_from((i, j) for i, j, _ in bonds)
    pos = {k: np.asarray(p) for k, p in nx.kamada_kawai_layout(G).items()}
    for i, j, order in bonds:
        p0, p1 = pos[i], pos[j]
        nrm = np.array([p0[1] - p1[1], p1[0] - p0[0]]) / (np.linalg.norm(p1 - p0) + 1e-9) * 0.035
        lines = {1: [(0, "-")], 1.5: [(0, "-"), (1, ":")], 2: [(-0.5, "-"), (0.5, "-")],
                 3: [(-1, "-"), (0, "-"), (1, "-")]}[order]
        for off, ls in lines:
            ax.plot([p0[0] + off * nrm[0], p1[0] + off * nrm[0]], [p0[1] + off * nrm[1], p1[1] + off * nrm[1]],
                    color="0.3", ls=ls, lw=1.2, zorder=1)
    for k, (el, _) in enumerate(atoms):
        ax.scatter(*pos[k], s=230, color=ELEMENT_COLOURS[el], edgecolor="k", lw=0.6, zorder=2)
        ax.text(*pos[k], el, ha="center", va="center", fontsize=7, zorder=3)
    ax.set_title(title, fontsize=8); ax.set_aspect("equal"); ax.axis("off")


fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
for ax, k in zip(axes, [1, 2, 6]):
    draw_molecule(*graphs[k], ax, f"{mols.smiles[k]}\n{mols.scaffold[k]} · pIC50 = {mols.pIC50[k]:.2f}")
fig.suptitle("SIMULATED molecules as graphs (hydrogens implicit)", fontsize=10)
plt.tight_layout(); plt.show()

**Atoms → feature vectors, bonds → adjacency.** Each atom gets 15 numbers: its element (one-hot over C, N, O, S, F, Cl),
whether it is aromatic, its number of heavy-atom neighbours (one-hot 1–4) and its number of attached hydrogens (one-hot
0–3; hydrogens are implicit, so this is where "is this O an OH or a C=O?" is stored). Molecules have different numbers
of atoms, so we **pad** every molecule to the largest size in the library and keep a **mask** that says which rows are
real atoms. Dense, padded batches are wasteful for big graphs but perfectly fine for small molecules.

**The network.** Three message-passing layers with sum aggregation, then a **sum readout** over the atoms and a small
MLP:

$$h_i^{(\ell+1)}=\mathrm{ReLU}\Big(W_{\text{self}}^{(\ell)}h_i^{(\ell)}+W_{\text{nbr}}^{(\ell)}\sum_{j\in\mathcal N(i)}h_j^{(\ell)}\Big),\qquad
\hat y=\mathrm{MLP}\Big(\sum_{i}h_i^{(3)}\Big).$$

*In words:* each atom combines its own state and the *sum* of its bonded neighbours' states, three times (so it sees
atoms up to three bonds away); then the atom vectors are added up into one molecule vector, which an MLP maps to the
predicted pIC50. Keeping self and neighbours apart (two weight matrices) lets the model tell "I am an O" from "my
neighbour is an O".

⚠️ **Why sum pooling?** The sum of atom vectors is permutation-invariant (Section 1) *and* remembers counts: a molecule
with two hydroxyls gets twice the hydroxyl signal. A mean would make a molecule look like its own dimer (same
proportions); a max would ignore how many times a group occurs. Xu *et al.* (2019) proved that sum aggregation is the
most expressive of the three.

In [ ]:
# ── 4.3 Featurise molecules: padded atom features, adjacency and mask ──
ATOM_ELEMENTS = ["C", "N", "O", "S", "F", "Cl"]
VALENCE_OF = {"C": 4, "N": 3, "O": 2}                         # used to count implicit hydrogens
N_MAX = max(len(a) for a, _ in graphs)


def featurize_molecule(atoms, bonds, n_max=N_MAX):
    """Return X (n_max × 15 atom features), A (n_max × n_max adjacency) and a 0/1 mask of real atoms."""
    n = len(atoms)
    bond_sum, degree = np.zeros(n), np.zeros(n, int)
    A = np.zeros((n_max, n_max), np.float32)
    for i, j, order in bonds:
        A[i, j] = A[j, i] = 1
        bond_sum[[i, j]] += order
        degree[[i, j]] += 1
    X = np.zeros((n_max, 15), np.float32)
    for k, (el, aromatic) in enumerate(atoms):
        X[k, ATOM_ELEMENTS.index(el)] = 1
        X[k, 6] = aromatic
        X[k, 6 + min(degree[k], 4)] = 1                       # columns 7–10: 1–4 heavy neighbours
        n_h = max(0, int(np.floor(VALENCE_OF.get(el, 0) - bond_sum[k] + 1e-9)))
        X[k, 11 + min(n_h, 3)] = 1                            # columns 11–14: 0–3 hydrogens
    mask = np.zeros(n_max, np.float32)
    mask[:n] = 1
    return X, A, mask


feats = [featurize_molecule(a, b) for a, b in graphs]
X_mol = torch.tensor(np.stack([f[0] for f in feats]))
A_mol = torch.tensor(np.stack([f[1] for f in feats]))
M_mol = torch.tensor(np.stack([f[2] for f in feats]))
y_mol = mols.pIC50.values.astype(np.float32)
cols = ATOM_ELEMENTS + ["arom"] + [f"deg{d}" for d in range(1, 5)] + [f"H{h}" for h in range(4)]
print("tensors:", tuple(X_mol.shape), tuple(A_mol.shape), tuple(M_mol.shape))
print(f"atom features of {mols.smiles[0]} (first {len(graphs[0][0])} rows are real atoms):")
print(pd.DataFrame(X_mol[0, :len(graphs[0][0])].numpy().astype(int), columns=cols).to_string())

In [ ]:
# ── 4.4 A message-passing network for molecules, and its training loop ──
class MolecularMPNN(nn.Module):
    """h_i ← ReLU(W_self h_i + W_nbr Σ_j A_ij h_j), repeated; sum readout; MLP → one number per molecule."""

    def __init__(self, d_in=15, d_hidden=64, n_layers=3):
        super().__init__()
        self.embed = nn.Linear(d_in, d_hidden)
        self.self_lin = nn.ModuleList(nn.Linear(d_hidden, d_hidden) for _ in range(n_layers))
        self.nbr_lin = nn.ModuleList(nn.Linear(d_hidden, d_hidden, bias=False) for _ in range(n_layers))
        self.head = nn.Sequential(nn.Linear(d_hidden, d_hidden), nn.ReLU(), nn.Linear(d_hidden, 1))

    def graph_embedding(self, X, A, mask):
        H = F.relu(self.embed(X)) * mask[..., None]           # padded rows are kept at zero
        for s_lin, n_lin in zip(self.self_lin, self.nbr_lin):
            H = F.relu(s_lin(H) + A @ n_lin(H)) * mask[..., None]
        return H.sum(dim=1)                                    # SUM readout: permutation-invariant, counts groups

    def forward(self, X, A, mask):
        return self.head(self.graph_embedding(X, A, mask)).squeeze(-1)


def train_mpnn(train_idx, epochs=40, batch=64, seed=SEED):
    """Mini-batch Adam on standardised pIC50; returns (model, predictions for every molecule)."""
    torch.manual_seed(seed)
    g = np.random.default_rng(seed)
    model = MolecularMPNN()
    opt = torch.optim.Adam(model.parameters(), lr=2e-3, weight_decay=1e-3)
    mu, sd = y_mol[train_idx].mean(), y_mol[train_idx].std()
    y_t = torch.tensor((y_mol - mu) / sd)
    for _ in range(epochs):
        model.train()
        perm = g.permutation(train_idx)
        for b in range(0, len(perm), batch):
            ib = torch.tensor(perm[b:b + batch])
            opt.zero_grad()
            F.mse_loss(model(X_mol[ib], A_mol[ib], M_mol[ib]), y_t[ib]).backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        return model, model(X_mol, A_mol, M_mol).numpy() * sd + mu


print("parameters:", sum(p.numel() for p in MolecularMPNN().parameters()))

**Baselines and splits.** The classical representation is a **circular fingerprint**: 1,024 bits, each set when a
particular atom environment (an atom plus its neighbours up to two bonds away) occurs somewhere in the molecule
(`molecular_fingerprint`, an ECFP-like hash). We fit ridge regression ([Module 07](07_ridge_lasso_sparse_regression.ipynb))
and a random forest ([Module 10](10_decision_trees_random_forests.ipynb)) on the bits.

We evaluate two ways ([Module 23](23_model_selection.ipynb)): a **random split** (test molecules resemble training
molecules: every scaffold is seen in training) and a **scaffold split** (all molecules of some scaffolds are held out —
the realistic question "will the model work on a new chemical series?"). The **noise ceiling** is the $R^2$ of the
oracle's noise-free potencies against the measured ones: no model can systematically beat it.

In [ ]:
# ── 4.5 🧪 Ground-truth check: MPNN vs fingerprint models, random vs scaffold split ──
t0 = time.time()
fp = np.array([molecular_fingerprint(a, b) for a, b in graphs])
parse_subs = lambda s: {int(p.split(":")[0][4:]): p.split(":")[1] for p in s.split("; ")}
y_true = np.array([lib["oracle"](sc, parse_subs(s))[0] for sc, s in zip(mols.scaffold, mols.substituents)])
perm = np.random.default_rng(SEED).permutation(len(mols))
tr_s, te_s = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED).split(mols, groups=mols.scaffold))
splits = {"random": (np.sort(perm[400:]), np.sort(perm[:400])), "scaffold": (tr_s, te_s)}
print("scaffolds held out in the scaffold split:", sorted(mols.scaffold.iloc[te_s].unique()), f"({len(te_s)} molecules)")
rows, mpnn_pred = [], {}
for split, (tr, te) in splits.items():
    preds = {"fingerprint + ridge": Ridge(alpha=10.0).fit(fp[tr], y_mol[tr]).predict(fp[te]),
             "fingerprint + random forest": RandomForestRegressor(200, max_features=0.3, random_state=SEED)
             .fit(fp[tr], y_mol[tr]).predict(fp[te])}
    mpnn_model, p_all = train_mpnn(tr)
    preds["MPNN (graph)"], mpnn_pred[split] = p_all[te], (p_all, mpnn_model)
    preds["noise ceiling (oracle)"] = y_true[te]
    for name, p in preds.items():
        rows.append(dict(split=split, model=name, R2=r2_score(y_mol[te], p),
                         RMSE=mean_squared_error(y_mol[te], p) ** 0.5))
res_mol = pd.DataFrame(rows).pivot(index="model", columns="split", values=["R2", "RMSE"]).round(3)
print(res_mol); print(f"({time.time() - t0:.0f} s)")

In [ ]:
# ── 4.6 Predicted vs measured potency on held-out molecules ──
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharex=True, sharey=True)
for ax, split in zip(axes, ["random", "scaffold"]):
    te = splits[split][1]
    p = mpnn_pred[split][0][te]
    for k, sc in enumerate(sorted(mols.scaffold.iloc[te].unique())):
        m = mols.scaffold.values[te] == sc
        ax.scatter(y_mol[te][m], p[m], s=8, alpha=0.6, label=sc if split == "scaffold" else None)
    ax.plot([2.5, 9], [2.5, 9], "k--", lw=0.8)
    ax.set_title(f"MPNN, {split} split: R² = {r2_score(y_mol[te], p):.2f}")
    ax.set_xlabel("measured pIC50 (SIMULATED)")
axes[0].set_ylabel("predicted pIC50"); axes[1].legend(fontsize=7, title="held-out scaffold")
plt.tight_layout(); plt.show()

🔍 **What just happened?** On the **random split** the MPNN reaches $R^2\approx0.6$, a little better than the two
fingerprint models ($\approx0.55$), but all three stay far below the noise ceiling ($\approx0.9$). The hidden rules
are defined relative to each scaffold's attachment sites ("a donor at site 0"), so a graph model must learn them
separately for each ring system from a few dozen examples each — a genuinely hard learning problem with 1,600
molecules. On the **scaffold split** every model drops sharply — the MPNN to about $R^2\approx0.2$ and the fingerprint
models to roughly zero or below (near, or worse than, just predicting the test mean): the held-out scaffolds have their
own baseline potency, which nothing in the training data reveals. (Exactly how far each model falls depends on *which*
scaffolds are held out — change the seed in the split and watch it move.) The scatter plot shows the typical signature — each held-out scaffold forms a cloud shifted
away from the diagonal, even where the ranking *within* the scaffold is partly right.

⚠️ **Which number would you put in a paper?** The random-split $R^2$ answers "how well do we interpolate within known
chemistry?"; the scaffold-split result answers "how well do we extrapolate to new chemistry?" — usually the question
drug discovery actually asks. Reporting only the random split is one of the commonest ways molecular ML papers
overstate performance; [Module 40](40_molecular_ml_drug_discovery.ipynb) studies splits, applicability domains and
activity cliffs in depth.

✏️ **Your turn.** Replace `H.sum(dim=1)` by `H.sum(dim=1) / mask.sum(1, keepdim=True)` (mean pooling) in
`graph_embedding` and re-run 4.5. Does the random-split $R^2$ change? Which molecular properties would you expect to be
hurt most by mean pooling (think of molecular weight)?

### 📄 Research connection — GNNs in biology and drug discovery

- **What the field does.** Graph learning entered drug discovery through molecular property prediction: message-passing networks on
molecular graphs (Gilmer *et al.* 2017, on the QM9 quantum-chemistry set) and their directed-edge variant D-MPNN,
used by Stokes *et al.* (2020) to discover the antibiotic halicin. On interaction networks, Decagon (Zitnik,
Agrawal & Leskovec 2018) predicted polypharmacy side effects with a relational GCN on a drug–protein graph; DeepFRI
(Gligorijević *et al.* 2021) predicted protein function with graph convolutions over residue-contact maps. GNNs
are also standard for spatial omics (cells as nodes, neighbours as edges; Module 33), knowledge graphs for drug
repurposing and gene-regulatory network inference.
- **What this notebook builds.** A dense GCN on a simulated interaction network (Section 2) and a sum-pooling MPNN on simulated molecules
(Section 4), both from scratch, with the hidden truth used to score them. Production models add edge features
(bond types, distances), directed messages, residual connections, uncertainty estimates and much larger,
curated data sets; libraries such as PyTorch Geometric and DGL provide sparse message passing.
- **Where the frontier is.** As of 2025, three issues dominate: (i) **generalisation** to new scaffolds, new proteins and new cell types,
for which honest splits often show much smaller gains over fingerprints than random splits suggest; (ii) the
**biases of interaction networks** (well-studied proteins have many more recorded partners, so a GNN can learn
"fame" instead of function); and (iii) combining graphs with 3-D structure and with pretrained representations
(protein and molecular language models, Modules 26 and 37).
- **Read next.** Stokes *et al.* 2020 (*Cell*), *A deep learning approach to antibiotic discovery*; Zitnik, Agrawal & Leskovec
2018 (*Bioinformatics*), *Modeling polypharmacy side effects with graph convolutional networks*; Gligorijević
*et al.* 2021 (*Nat Commun*), *Structure-based protein function prediction using graph convolutional networks*.

---
## 5 · What message passing cannot see: the Weisfeiler–Lehman limit

> 📍 **Where we are.** Message passing only ever looks at neighbourhoods. Are there different graphs that *every*
> message-passing GNN must confuse? Xu *et al.* (2019) and Morris *et al.* (2019) answered this with a 1968 algorithm
> for testing whether two graphs are the same.

**1-WL colour refinement** (Weisfeiler & Leman 1968, in the form used by Xu *et al.* 2019). Give every node the same
colour. Then repeat: each node's new colour is a fresh code for the pair (its own colour, the *multiset* of its
neighbours' colours). Two graphs that end with different colour **histograms** are certainly different; if the
histograms always match, 1-WL cannot tell them apart. Xu *et al.* proved that a message-passing GNN is **at most as
powerful as 1-WL**: it, too, builds each node's representation from its colour and its neighbours' multiset, so any two
graphs 1-WL cannot separate produce identical node representations and, after a sum/mean/max readout, identical graph
predictions — no matter how it is trained.

In [ ]:
# ── 5.1 1-WL colour refinement, and two graphs it cannot distinguish ──
def wl_refine(A, n_iter=3):
    """Weisfeiler–Lehman colour refinement. Returns the list of colour arrays after 0, 1, ... iterations."""
    colours = np.zeros(len(A), int)                      # start: every node the same colour
    history = [colours.copy()]
    for _ in range(n_iter):
        signatures = [(colours[v], tuple(sorted(colours[u] for u in np.flatnonzero(A[v])))) for v in range(len(A))]
        code = {s: i for i, s in enumerate(sorted(set(signatures)))}     # a fresh colour per distinct signature
        colours = np.array([code[s] for s in signatures])
        history.append(colours.copy())
    return history


def ring_graph(n):
    """Adjacency of a single n-membered ring (e.g. benzene for n = 6)."""
    A = np.zeros((n, n))
    for i in range(n):
        A[i, (i + 1) % n] = A[(i + 1) % n, i] = 1
    return A


def two_triangles():
    """Adjacency of two disconnected triangles (6 nodes, also 2-regular)."""
    A = np.zeros((6, 6))
    for base in (0, 3):
        for i, j in [(0, 1), (1, 2), (2, 0)]:
            A[base + i, base + j] = A[base + j, base + i] = 1
    return A


for name, A in [("hexagon (one 6-ring) ", ring_graph(6)), ("two triangles       ", two_triangles())]:
    hist = wl_refine(A)
    print(f"{name}: colour histograms after 0,1,2,3 rounds =", [tuple(np.bincount(c)) for c in hist])

In [ ]:
# ── 5.2 A random (untrained) MPNN gives these two graphs the SAME embedding ──
def tiny_mpnn_embedding(A, node_features, seed=SEED, d=8):
    """Two message-passing layers + sum readout, random weights; returns one vector per graph."""
    torch.manual_seed(seed)
    s1, n1 = nn.Linear(node_features.shape[1], d), nn.Linear(node_features.shape[1], d, bias=False)
    s2, n2 = nn.Linear(d, d), nn.Linear(d, d, bias=False)
    At, H = torch.tensor(A, dtype=torch.float32), torch.tensor(node_features, dtype=torch.float32)
    H = F.relu(s1(H) + At @ n1(H))
    H = F.relu(s2(H) + At @ n2(H))
    return H.sum(0).detach().numpy()


ones6 = np.ones((6, 1), np.float32)                      # all atoms identical (e.g. all carbons)
e_hex, e_tri = tiny_mpnn_embedding(ring_graph(6), ones6), tiny_mpnn_embedding(two_triangles(), ones6)
print(f"‖MPNN(hexagon) − MPNN(two triangles)‖ = {np.linalg.norm(e_hex - e_tri):.2e}  → identical")

# The fix: give each atom a structural feature 1-WL/MPNN cannot compute itself — e.g. how many triangles it sits in
triangles = lambda A: (np.linalg.matrix_power(A, 3).diagonal() / 2).astype(int)   # closed 3-walks / 2
print("triangles per node — hexagon:", triangles(ring_graph(6)), " two triangles:", triangles(two_triangles()))
feat_hex = np.c_[ones6, triangles(ring_graph(6))[:, None]].astype(np.float32)
feat_tri = np.c_[ones6, triangles(two_triangles())[:, None]].astype(np.float32)
d_fix = np.linalg.norm(tiny_mpnn_embedding(ring_graph(6), feat_hex) - tiny_mpnn_embedding(two_triangles(), feat_tri))
print(f"with a triangle-count feature: ‖difference‖ = {d_fix:.2f}  → now distinguishable")

🔍 **What just happened?** 1-WL gives the hexagon and the two-triangles graph the same colour histogram at every round
(both are 2-regular: every node has two neighbours, forever), and an untrained MPNN gives them byte-for-byte identical
graph embeddings. Yet they are different molecules — benzene versus two fragments. The cure is to hand the network a
**structural feature message passing cannot derive on its own**, here the number of triangles each atom belongs to (1
in the triangles, 0 in the hexagon), after which the embeddings differ. This is why real molecular GNNs include ring
membership, ring size and related descriptors in their atom features, and why "more expressive" GNNs exist: higher-order
networks that pass messages between *pairs* of nodes (Morris *et al.* 2019), and **positional encodings** from Laplacian
eigenvectors (Section 1; Dwivedi *et al.* 2023) that give symmetric nodes distinct coordinates.

⚠️ **Expressivity is necessary, not sufficient.** A maximally expressive GNN can still generalise badly (Section 4). The
WL hierarchy tells you what a model *could* distinguish in principle, not what it *will* learn from finite, noisy data.

✏️ **Your turn.** Use `wl_refine` on a 4-ring (`ring_graph(4)`) versus two disconnected edges (a graph with adjacency
linking 0–1 and 2–3). Do their colour histograms differ? What structural feature would separate them?

### 📄 Research connection — Graph convolutions, GraphSAGE, GAT, GIN — and their expressive power

- **What the field does.** The modern wave began with spectral graph convolutions (Bruna *et al.* 2014; Defferrard *et al.* 2016) and their
simplification to the GCN (Kipf & Welling 2017). GraphSAGE (Hamilton, Ying & Leskovec 2017) made learning
*inductive* by sampling and aggregating neighbourhoods; GAT (Veličković *et al.* 2018) weighted neighbours with
attention; Gilmer *et al.* (2017) unified them as message passing; GIN (Xu *et al.* 2019) matched the 1-WL test's
power with sum aggregation and analysed expressivity.
- **What this notebook builds.** From-scratch GCN, label propagation, generic message passing and a sum-pooling MPNN, plus the 1-WL test and a
concrete pair of graphs no MPNN can separate. The toolkit's `molecular_fingerprint` is the pre-deep-learning
baseline these models are measured against.
- **Where the frontier is.** As of 2025: provably more expressive architectures (higher-order / subgraph GNNs), graph transformers with
positional encodings, and a lively debate on when the extra expressivity actually improves biological predictions
versus when a well-featurised message-passing (1-WL) model or even a fingerprint baseline is enough. Over-smoothing
and over-squashing remain open at the architectural level.
- **Read next.** Kipf & Welling 2017 (*ICLR*), *Semi-supervised classification with graph convolutional networks*; Hamilton, Ying
& Leskovec 2017 (*NeurIPS*), *Inductive representation learning on large graphs* (GraphSAGE); Xu, Hu, Leskovec &
Jegelka 2019 (*ICLR*), *How powerful are graph neural networks?* (GIN).

---
## 6 · Attention on graphs, and why a transformer is a GNN

> 📍 **Where we are.** A GCN averages all neighbours with fixed, degree-based weights. But some neighbours matter more:
> a substrate for its kinase, a co-complex member for a protein. **Graph attention** lets a node *learn* how much to
> listen to each neighbour.

The original graph attention network (**GAT**, Veličković *et al.* 2018) scores a neighbour *additively*,
$e_{vu}=\mathrm{LeakyReLU}\big(a^\top[Wh_v\,\Vert\,Wh_u]\big)$ (a small vector $a$ applied to the concatenated,
transformed features), then softmaxes $e_{vu}$ over the neighbours. We implement instead the **transformer-style
scaled dot-product** variant (Vaswani *et al.* 2017; used by graph transformers), which is the exact mechanism of
[Module 26](26_attention_transformers.ipynb) — each node computes a **query** and every neighbour a **key** and a
**value**, the weight is a softmax of query·key, and the output is the attention-weighted sum of the values:

$$\alpha_{vu}=\frac{\exp\!\big(q_v^\top k_u/\sqrt{d}\big)}{\sum_{w\in\mathcal N(v)\cup\{v\}}\exp\!\big(q_v^\top k_w/\sqrt{d}\big)},\qquad
h_v'=\sum_{u\in\mathcal N(v)\cup\{v\}}\alpha_{vu}\,V h_u .$$

*In words:* node $v$ scores each of its neighbours by how well its query matches their key, turns the scores into
weights that sum to one (softmax), and mixes the neighbours' values with those weights. A neighbour the node finds
irrelevant gets a weight near zero — the layer can *refuse to average*, which is exactly what heterophilous graphs
(Section 2) need. This is message passing with a learned, data-dependent aggregation.

**The connection to transformers.** If every node attends to *every* other node — the graph is **complete** — this is
precisely self-attention, the transformer layer. So a transformer is a graph attention network on the complete graph,
and a GNN on a sparse graph is a transformer that is only allowed to attend along edges. The adjacency is an
**attention mask**. That single idea unifies Modules 26 and 28, and it is why "graph transformers" (attend to all
nodes, but add the graph as a bias) are a natural and popular design.

In [ ]:
# ── 6.1 Graph attention from scratch (dense, single head) ──
class GraphAttentionLayer(nn.Module):
    """One attention layer. mask[v, u] = 1 if u may send to v (neighbours + self); others get weight 0."""

    def __init__(self, d_in, d_out, d_key=16):
        super().__init__()
        self.q = nn.Linear(d_in, d_key, bias=False)
        self.k = nn.Linear(d_in, d_key, bias=False)
        self.v = nn.Linear(d_in, d_out)
        self.scale = d_key ** 0.5

    def forward(self, H, mask):
        scores = self.q(H) @ self.k(H).T / self.scale
        scores = scores.masked_fill(mask == 0, float("-inf"))   # cannot attend where there is no edge
        alpha = torch.softmax(scores, dim=1)
        return alpha @ self.v(H), alpha


class GAT(nn.Module):
    def __init__(self, d_in, d_hidden, d_out, dropout=0.5):
        super().__init__()
        self.l1 = GraphAttentionLayer(d_in, d_hidden)
        self.l2 = GraphAttentionLayer(d_hidden, d_out)
        self.dropout = dropout

    def forward(self, H, mask):
        H, a1 = self.l1(F.dropout(H, self.dropout, self.training), mask)
        H, a2 = self.l2(F.dropout(F.relu(H), self.dropout, self.training), mask)
        return H, (a1, a2)


def train_gat(A, X, y, train_idx, epochs=200, seed=SEED):
    torch.manual_seed(seed)
    mask = torch.tensor(A + np.eye(len(A)), dtype=torch.float32)       # neighbours + self
    Xt, yt, tr = torch.tensor(X, dtype=torch.float32), torch.tensor(y), torch.tensor(train_idx)
    model = GAT(X.shape[1], 16, int(y.max()) + 1)
    opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        F.cross_entropy(model(Xt, mask)[0][tr], yt[tr]).backward(); opt.step()
    model.eval()
    with torch.no_grad():
        out, (a1, a2) = model(Xt, mask)
    return out.argmax(1).numpy(), a2.numpy()

In [ ]:
# ── 6.2 🧪 Ground-truth check: does the attention concentrate on same-module neighbours? ──
t0 = time.time()
gat_pred, attn = train_gat(A_ppi, X_ppi, y_ppi, labelled)
print(f"GAT accuracy on unlabelled proteins: {np.mean(gat_pred[unlabelled] == y_ppi[unlabelled]):.3f}  "
      f"(GCN was {np.mean(pred['GCN (features + graph)'][unlabelled] == y_ppi[unlabelled]):.3f})")

same = (y_ppi[:, None] == y_ppi[None, :]) & (A_ppi > 0)               # edges within a module
diff = (y_ppi[:, None] != y_ppi[None, :]) & (A_ppi > 0)               # edges between modules
uniform = (A_ppi + np.eye(600)) / (A_ppi + np.eye(600)).sum(1, keepdims=True)
print(f"mean attention weight on a same-module neighbour: {attn[same].mean():.4f}")
print(f"mean attention weight on a cross-module neighbour: {attn[diff].mean():.4f}")
print(f"(a degree-blind GCN would give every neighbour ≈ {uniform[A_ppi > 0].mean():.4f})")
print(f"share of each node's incoming attention spent on same-module edges: {(attn * same).sum() / (attn * (same | diff)).sum():.2f}"
      f"  (same-module edges are {same.sum() / (same | diff).sum():.2f} of all edges)")
print(f"({time.time() - t0:.0f} s)")

🔍 **What just happened?** Trained only on the 60 labels, the attention layer puts **more weight on same-module
neighbours** (≈ 0.114) than on cross-module ones (≈ 0.077), so that about **80 % of each node's attention mass flows
along same-module edges even though those make up only 73 % of its edges** — it has discovered, without being told, that
partners in the same module are the informative ones. On this strongly homophilous network GAT and GCN reach similar
accuracy; the advantage of attention grows on **heterophilous or noisy** graphs, where the ability to down-weight
misleading neighbours matters (as in your ✏️ turn from Section 2).

⚠️ **Attention is not explanation.** A high attention weight means "this neighbour influenced the representation here",
not "this edge is biologically causal". Attention maps are hypotheses to test, like a learned network edge
([Module 17](17_bayesian_networks_inference_learning.ipynb)), not mechanisms.

✏️ **Your turn.** Build the attention mask for the *complete* graph (`mask = torch.ones(600, 600)`) and retrain. The
model may now attend to any protein, not just partners — does accuracy go up or down, and why might ignoring the known
interaction graph hurt here?

---
## 7 · Geometry: invariance and equivariance in 3-D

> 📍 **Where we are.** Atoms and residues live in 3-D space. A molecule's energy, a protein's fold and a binding
> affinity do not change if we rotate or translate the whole structure — but the raw coordinates change completely.
> **Geometric deep learning** (Bronstein *et al.* 2021) is about building models that respect such symmetries.

Two words we must keep apart:

* A function $f$ is **invariant** under a transformation $g$ if $f(g\cdot x)=f(x)$: the output does not move. A
  molecule's predicted potency should be invariant to rotation.
* A function $f$ is **equivariant** if $f(g\cdot x)=g\cdot f(x)$: the output moves *with* the input. Predicted
  atom *forces* or *coordinates* should rotate when the molecule rotates — a force is an arrow attached to space.

The group of rotations, reflections and translations in 3-D is called $E(3)$; without reflections, $SE(3)$. Permutation
symmetry (Section 1) and these spatial symmetries together are the symmetries of a molecule or a protein structure.

**The task.** Classify a protein backbone as a **3-, 4- or 5-helix bundle** from its Cα coordinates. `simulate_helix_bundle`
gives exact coordinates; we set helix lengths so that every bundle has 56 residues (so a coordinate-reading model cannot
cheat on size). Then we present each structure in a **random orientation** (a random rotation and translation) at test
time — just as a real structure has no canonical frame — and compare three strategies.

In [ ]:
# ── 7.1 SIMULATED helical bundles in random 3-D orientations ──
HELIX_LEN = {3: 16, 4: 11, 5: 8}                             # chosen so 3,4,5 helices all give 56 Cα atoms


def random_rotation(rng):
    """A uniformly random proper rotation matrix (det = +1) via QR of a Gaussian matrix."""
    Q, R = np.linalg.qr(rng.normal(size=(3, 3)))
    Q = Q * np.sign(np.diag(R))                              # fix signs so Q is a rotation
    if np.linalg.det(Q) < 0:
        Q[:, 0] = -Q[:, 0]
    return Q


def make_bundles(rng, n_per_class, noise=1.0, rotate=True):
    """n_per_class structures of each type (3,4,5 helices); optionally in a random orientation. Returns coords, labels."""
    coords, labels = [], []
    for k in (3, 4, 5):
        for _ in range(n_per_class):
            c = simulate_helix_bundle(k, HELIX_LEN[k], 4, rng=rng, noise=noise)["coords"]
            if rotate:
                c = c @ random_rotation(rng).T + rng.uniform(-20, 20, 3)   # rotate + translate the whole structure
            coords.append(c); labels.append(k - 3)
    return np.array(coords), np.array(labels)


rng = np.random.default_rng(SEED + 7)
Xc, yc = make_bundles(rng, 200, rotate=False)               # training set in a canonical orientation
Xtest, ytest = make_bundles(rng, 200, rotate=True)          # test set in fresh random orientations
print("structures:", Xc.shape, "→ 600 train, 600 test; every bundle has", Xc.shape[1], "Cα atoms")

fig = plt.figure(figsize=(11, 3.2))
for i, k in enumerate((3, 4, 5)):
    ax = fig.add_subplot(1, 3, i + 1, projection="3d")
    c = simulate_helix_bundle(k, HELIX_LEN[k], 4)["coords"]
    ax.plot(c[:, 0], c[:, 1], c[:, 2], "-", color="0.6", lw=1)
    ax.scatter(c[:, 0], c[:, 1], c[:, 2], c=np.arange(len(c)), cmap="viridis", s=12)
    ax.set_title(f"SIMULATED {k}-helix bundle"); ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
plt.tight_layout(); plt.show()

In [ ]:
# ── 7.2 Three strategies: raw coordinates, rotation augmentation, and rotation-invariant features ──
def pairwise_distances(coords):
    """Flatten the upper triangle of each structure's Cα–Cα distance matrix: a rotation/translation-invariant feature."""
    D = np.linalg.norm(coords[:, :, None, :] - coords[:, None, :, :], axis=-1)
    iu = np.triu_indices(coords.shape[1], 1)
    return D[:, iu[0], iu[1]]


def train_structure_mlp(Xtr, ytr, Xte, yte, epochs=150, seed=SEED, d_hidden=64):
    """A plain MLP classifier (3 classes). Returns (train accuracy, test accuracy)."""
    torch.manual_seed(seed)
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6
    Xt, Xe = torch.tensor((Xtr - mu) / sd, dtype=torch.float32), torch.tensor((Xte - mu) / sd, dtype=torch.float32)
    yt = torch.tensor(ytr)
    model = nn.Sequential(nn.Linear(Xt.shape[1], d_hidden), nn.ReLU(),
                          nn.Linear(d_hidden, d_hidden), nn.ReLU(), nn.Linear(d_hidden, 3))
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    for _ in range(epochs):
        opt.zero_grad(); F.cross_entropy(model(Xt), yt).backward(); opt.step()
    with torch.no_grad():
        return (model(Xt).argmax(1).numpy() == ytr).mean(), (model(Xe).argmax(1).numpy() == yte).mean()


rng = np.random.default_rng(SEED + 9)
Xaug, yaug = make_bundles(rng, 200, rotate=True)            # training set WITH random orientations (augmentation)
t0 = time.time()
tr_raw, te_raw = train_structure_mlp(Xc.reshape(600, -1), yc, Xtest.reshape(600, -1), ytest)
tr_aug, te_aug = train_structure_mlp(Xaug.reshape(600, -1), yaug, Xtest.reshape(600, -1), ytest)
tr_dist, te_dist = train_structure_mlp(pairwise_distances(Xc), yc, pairwise_distances(Xtest), ytest)
print("🧪 Ground-truth check — accuracy on 600 freshly ROTATED test structures:")
print(f"   (a) MLP on raw coords, trained canonical : train {tr_raw:.2f}  test {te_raw:.2f}")
print(f"   (b) MLP on raw coords, trained augmented : train {tr_aug:.2f}  test {te_aug:.2f}")
print(f"   (c) MLP on pairwise distances (invariant): train {tr_dist:.2f}  test {te_dist:.2f}")
print(f"({time.time() - t0:.0f} s)")

🔍 **What just happened?** Model (a) fits the training structures perfectly but **fails on rotated test structures**
(accuracy near the 1/3 chance level): it memorised coordinate patterns that a rotation destroys. Model (b), trained with
random rotations, generalises well — **data augmentation** teaches the symmetry approximately, at the cost of needing
more data and never being exact. Model (c), which feeds the network only **pairwise Cα–Cα distances**, is *exactly*
rotation- and translation-invariant by construction (distances do not change when you rotate or move a rigid body) and
reaches high accuracy with the *canonical* training set and no augmentation at all.

This is the core lesson of geometric deep learning: **bake the symmetry into the representation or the architecture**
rather than hoping the model learns it. Distances are the simplest invariant; the next section builds a layer that is
invariant in its features *and* equivariant in its coordinates, so that it can also *move* atoms — the operation a
structure-prediction or docking model needs.

⚠️ **Invariant features can lose information.** Pairwise distances cannot distinguish a structure from its mirror image
(chirality), which matters for drugs: two mirror-image molecules can have very different biological effects. $SE(3)$
-equivariant models and signed dihedral features keep handedness; plain distance features throw it away.

✏️ **Your turn.** Make the test set harder: in 7.2 train model (b) with only 20 structures per class
(`make_bundles(rng, 20, rotate=True)`) and compare its rotated-test accuracy with the distance model (c) trained on the
same 20 per class. Which strategy is more data-efficient?

---
## 8 · An E(n)-equivariant layer (EGNN) from scratch

> 📍 **Where we are.** Distances (Section 7) are invariant but static — they cannot *update* positions. AlphaFold's
> structure module, protein and small-molecule docking, and diffusion models for structure all need layers that
> transform 3-D coordinates **equivariantly**: rotate the input, and the output coordinates rotate with it. We implement
> the simplest such layer, the **E(n)-equivariant graph neural network** (Satorras, Hoogeboom & Welling 2021).

Each node has an invariant feature vector $h_i$ and a 3-D position $x_i$. One EGNN layer updates both:

$$m_{ij}=\phi_e\!\big(h_i,h_j,\lVert x_i-x_j\rVert^2\big),\quad
x_i'=x_i+\frac1{N-1}\sum_{j\ne i}(x_i-x_j)\,\phi_x(m_{ij}),\quad
h_i'=\phi_h\!\Big(h_i,\sum_{j\ne i}m_{ij}\Big).$$

*In words:* the message between $i$ and $j$ depends only on their features and their **squared distance** — a rotation-
and translation-invariant quantity — so the messages, and hence the updated features $h_i'$, are invariant. The position
update moves $x_i$ along the **relative vectors** $x_i-x_j$, each scaled by a learned number $\phi_x(m_{ij})$. Because a
rotation $R$ sends $x_i-x_j$ to $R(x_i-x_j)$ while leaving the distances (and so $\phi_x$) unchanged, the whole position
update rotates with the input: it is **equivariant**. No spherical harmonics, no special machinery — just "use distances
for features, relative vectors for positions". We verify both properties numerically to machine precision.

In [ ]:
# ── 8.1 One E(n)-equivariant layer ──
class EGNNLayer(nn.Module):
    """Satorras et al. 2021. Invariant features h (equivariant-safe) and equivariant coordinates x, on a dense graph."""

    def __init__(self, d_h, d_edge=16):
        super().__init__()
        self.phi_e = nn.Sequential(nn.Linear(2 * d_h + 1, d_edge), nn.SiLU(), nn.Linear(d_edge, d_edge), nn.SiLU())
        self.phi_x = nn.Sequential(nn.Linear(d_edge, d_edge), nn.SiLU(), nn.Linear(d_edge, 1))
        self.phi_h = nn.Sequential(nn.Linear(d_h + d_edge, d_h), nn.SiLU(), nn.Linear(d_h, d_h))

    def forward(self, h, x):
        B, N, _ = h.shape
        diff = x[:, :, None, :] - x[:, None, :, :]                 # relative vectors x_i − x_j  (B, N, N, 3)
        dist2 = (diff ** 2).sum(-1, keepdim=True) / 100.0          # squared distance (scaled to ~1 for stable training)
        hi = h[:, :, None, :].expand(B, N, N, -1)
        hj = h[:, None, :, :].expand(B, N, N, -1)
        m = self.phi_e(torch.cat([hi, hj, dist2], dim=-1))         # messages depend only on features + distance
        off = (1 - torch.eye(N, device=h.device))[None, :, :, None]   # drop self-messages (i = j)
        m = m * off
        x_new = x + (diff * self.phi_x(m) * off).sum(2) / (N - 1)  # move along relative vectors → equivariant
        h_new = h + self.phi_h(torch.cat([h, m.sum(2)], dim=-1))   # update features from pooled messages → invariant
        return h_new, x_new


# ── 8.2 🧪 Ground-truth check: rotating/translating the input rotates x and leaves h unchanged ──
torch.manual_seed(SEED)
layer = EGNNLayer(d_h=8).double()                                  # double precision for a clean numerical check
h0 = torch.randn(1, 12, 8, dtype=torch.float64)
x0 = torch.randn(1, 12, 3, dtype=torch.float64)
R = torch.tensor(random_rotation(np.random.default_rng(1)))        # a random rotation
t = torch.randn(3, dtype=torch.float64)                            # a random translation
h_a, x_a = layer(h0, x0)
h_b, x_b = layer(h0, x0 @ R.T + t)                                 # same layer on the moved structure
print(f"features invariant:   max|h(Rx+t) − h(x)|        = {(h_b - h_a).abs().max().item():.2e}")
print(f"coords equivariant:   max|x(Rx+t) − (R x(x)+t)|  = {(x_b - (x_a @ R.T + t)).abs().max().item():.2e}")

In [ ]:
# ── 8.3 An EGNN classifier: equivariance ⇒ correct on rotated structures with NO augmentation ──
class EGNNClassifier(nn.Module):
    """Invariant features start identical on every atom; one EGNN layer; mean readout; linear head (3 classes)."""

    def __init__(self, d_h=16):
        super().__init__()
        self.h0 = nn.Parameter(torch.randn(1, 1, d_h) * 0.1)
        self.layer = EGNNLayer(d_h)
        self.head = nn.Linear(d_h, 3)

    def forward(self, x):
        h = self.h0.expand(x.shape[0], x.shape[1], -1)
        h, x = self.layer(h, x)
        return self.head(h.mean(1))                                # invariant features → invariant prediction


rng = np.random.default_rng(SEED + 11)
Xsmall, ysmall = make_bundles(rng, 20, rotate=False)              # trained on only 20 CANONICAL structures per class
t0 = time.time()
torch.manual_seed(SEED)
clf = EGNNClassifier()
opt = torch.optim.Adam(clf.parameters(), lr=3e-3)
x_tr, y_tr = torch.tensor(Xsmall, dtype=torch.float32), torch.tensor(ysmall)
x_te = torch.tensor(Xtest, dtype=torch.float32)
for _ in range(60):
    opt.zero_grad(); F.cross_entropy(clf(x_tr), y_tr).backward(); opt.step()
with torch.no_grad():
    acc_tr = (clf(x_tr).argmax(1) == y_tr).float().mean().item()
    acc_te = (clf(x_te).argmax(1).numpy() == ytest).mean()
print(f"EGNN trained on 60 canonical structures → train {acc_tr:.2f}, ROTATED test {acc_te:.2f}")
print(f"(for comparison, the raw-coordinate MLP from 7.2 scored {te_raw:.2f} on rotated test structures)")
print(f"({time.time() - t0:.0f} s)")

🔍 **What just happened?** The symmetry check passes to about $10^{-16}$ (machine precision) — the features are invariant
and the coordinates equivariant, as designed, *before any training*. The EGNN classifier, trained on only 60 **canonical**
structures with no rotation augmentation, still classifies freshly **rotated** test structures well above chance, because
its predictions literally cannot depend on orientation. The raw-coordinate MLP, by contrast, was at chance on rotated
structures. Equivariance is data-augmentation's exactness without augmentation's cost.

> 🔗 **Builds on** [Module 22 · Neural networks](22_neural_networks.ipynb): weight tying in a CNN makes it
> *translation*-equivariant on a sequence (a motif is recognised wherever it occurs); an EGNN generalises the idea to
> *rotations and translations in 3-D*, and a GNN to *permutations on a graph*. Each architecture encodes a symmetry by
> sharing parameters across the positions the symmetry relates.

**Where this matters.** AlphaFold 2's structure module and most protein- and ligand-structure models are built from
$SE(3)$-equivariant operations; EGNN-style layers are used in molecular property prediction, in docking, and as the
denoiser inside diffusion models that *generate* 3-D structures ([Module 27](27_diffusion_flow_matching.ipynb);
[Module 39](39_generative_design.ipynb)). The same principle — respect the symmetry — recurs throughout geometric deep
learning.

✏️ **Your turn.** Add a reflection to the test transform (`R[:, 0] = -R[:, 0]`, making `det(R) = −1`) in 8.2 and re-run.
Is the EGNN still invariant? (It is $E(3)$-equivariant, so yes — it cannot tell a structure from its mirror image. What
would you add to the features to break that, if chirality mattered?)

### 📄 Research connection — Geometric deep learning and equivariance

- **What the field does.** Bronstein *et al.* (2021) framed CNNs, GNNs and transformers as one subject: learning functions that respect a
data domain's symmetry group. For 3-D molecular and protein data the relevant groups are $E(3)$/$SE(3)$ plus atom
permutations. SchNet (Schütt *et al.* 2017) used continuous-filter convolutions on interatomic distances;
Tensor-Field Networks and SE(3)-Transformers (Thomas *et al.* 2018; Fuchs *et al.* 2020) used spherical harmonics;
EGNN (Satorras *et al.* 2021) achieved equivariance with only distances and relative vectors. AlphaFold 2 (Jumper
*et al.* 2021) made such reasoning central to protein structure prediction.
- **What this notebook builds.** A from-scratch EGNN layer with a machine-precision check of invariance and equivariance, and a demonstration that
equivariance replaces rotation augmentation. The invariant-distance baseline of Section 7 is the simplest member of
this family.
- **Where the frontier is.** As of 2025: how much exact equivariance helps versus large models trained with heavy augmentation (AlphaFold 3 and
some diffusion models relax strict equivariance); scaling equivariant layers to large assemblies; equivariant
generative models for proteins and ligands; and handling chirality and periodic (crystal) symmetries correctly.
- **Read next.** Satorras, Hoogeboom & Welling 2021 (*ICML*), *E(n) equivariant graph neural networks*; Bronstein, Bruna, Cohen &
Veličković 2021 (*arXiv:2104.13478*), *Geometric deep learning*; Jumper *et al.* 2021 (*Nature*), *Highly accurate
protein structure prediction with AlphaFold*.

---
## 9 · 🧭 From toy to real data

> 📍 **Where we are.** Everything above ran in about two minutes on dense, simulated graphs with a known truth. Real
> graph learning is larger, sparser and messier. This section is reading, not code: what changes, what to use, and the
> first things that go wrong.

**Scale and sparsity.** Real graphs are big and sparse: STRING has tens of thousands of proteins, molecular libraries
run from thousands (a single assay) to billions (ZINC, Enamine REAL), and a citation or interaction graph can have
millions of nodes. Dense $n\times n$ matrices are impossible; production code stores an **edge list** and uses
sparse **scatter-add** message passing. For huge graphs you sample neighbourhoods (GraphSAGE) or subgraphs
(Cluster-GCN) rather than processing the whole graph at once.

**Tools.** [PyTorch Geometric](https://pytorch-geometric.readthedocs.io) and [DGL](https://www.dgl.ai) provide sparse
message passing and dozens of layers (GCN, GraphSAGE, GAT, GIN, MPNN, EGNN, …); for molecules,
[Chemprop](https://chemprop.readthedocs.io) (a directed-MPNN, as in Stokes *et al.* 2020),
[RDKit](https://www.rdkit.org) (real SMILES parsing, fingerprints, scaffolds), [DeepChem](https://deepchem.io) and the
[Therapeutics Data Commons](https://tdcommons.ai). We avoided all of these on purpose — the course installs none of them
— and wrote the layers by hand so the mechanism is visible.

**Public data.** Interaction networks: [STRING](https://string-db.org), [BioGRID](https://thebiogrid.org),
[IntAct](https://www.ebi.ac.uk/intact/). Molecules and assays: [ChEMBL](https://www.ebi.ac.uk/chembl/),
[PubChem](https://pubchem.ncbi.nlm.nih.gov), [MoleculeNet](https://moleculenet.org),
[QM9](http://quantum-machine.org/datasets/), [ZINC](https://zinc.docking.org). Structures:
[PDB](https://www.rcsb.org) and the [AlphaFold DB](https://alphafold.ebi.ac.uk).

**The first things that go wrong.**

1. **Leaky splits.** A random train/test split puts near-duplicate molecules (or homologous proteins, or the two halves
   of one complex) on both sides, inflating the score. Use **scaffold splits** for molecules (Section 4), sequence- or
   structure-identity clustering for proteins, and time splits to mimic prospective use.
2. **Network bias.** Interaction and annotation databases record far more about well-studied genes. A GNN can learn
   "this protein is famous" as a proxy for the label; strong degree- or study-bias baselines are essential, and a model
   that only beats them on popular nodes has learned the wrong thing.
3. **The graph is a noisy hypothesis.** Edges are incomplete and partly wrong (false positives in high-throughput
   screens, missing interactions), and homophily may not hold for your label (Section 2). A GNN inherits every bias of
   the graph you feed it; sometimes a careful featurised baseline without the graph wins.

Other real-world concerns: 3-D models must handle side chains, multiple conformations and protonation states, not just a
Cα trace; equivariant models can be slow; and, as throughout the course, a predicted edge, attention weight or binding
score is a hypothesis to be tested experimentally, not a fact.

---
## 📌 Key takeaways

- **Graphs are data, and node order is arbitrary.** A graph model must be **permutation-equivariant** (node outputs) or
  **-invariant** (graph outputs). GNNs get this for free by using one shared weight matrix and an order-independent
  aggregation; an MLP on the flattened adjacency does not.
- **A GCN layer is $\sigma(\hat A H W)$**: transform every node with shared weights, then average over neighbours. On a
  simulated protein network it beat the features-only MLP and graph-only label spreading (0.97 vs 0.69 and 0.72) — but
  **only when the graph is homophilous**; below a homophily of about 1/2 the graph hurt.
- **Every GNN is message passing** (message → aggregate → update), the same pattern as belief propagation in
  [Module 17](17_bayesian_networks_inference_learning.ipynb) with learned functions instead of probability tables.
  Stacking too many layers causes **over-smoothing**: all nodes converge to one vector, so most GNNs are shallow (2–5 layers).
- **Graph-level prediction needs a readout**; **sum pooling** is permutation-invariant and counts substructures. A
  from-scratch MPNN on simulated molecules edged out fingerprint baselines on a **random split** ($R^2\approx0.63$) but
  fell to $R^2\approx0.2$ on a **scaffold split** (fingerprint models ≈ 0 or below) — the honest test of generalisation
  to new chemistry.
- **Message passing is bounded by the 1-WL test**: there are non-isomorphic graphs (a hexagon vs two triangles) that no
  MPNN can distinguish. Structural features (ring/triangle counts), higher-order GNNs or Laplacian positional encodings
  restore the missing power.
- **Graph attention** learns how much to weight each neighbour (and concentrated on same-module partners here); a
  **transformer is a GNN on the complete graph**, with the adjacency as an attention mask.
- **Geometry needs symmetry built in.** For 3-D structures, rotation-**invariant** distance features or an
  **E(n)-equivariant** layer (EGNN: distances for features, relative vectors for positions) classify rotated structures
  well above chance with no augmentation (distances 1.00, EGNN 0.89); raw coordinates stay at chance. This is the principle behind AlphaFold's structure module.

---
## ✅ Checkpoint

1. Compute one GCN layer, $\hat A H W$ with ReLU, by hand for a 3-node path 1–2–3 with scalar features $h=(1,0,1)$ and $W=1$ (so one feature in, one out). Give the self-loop degrees and the output of node 2.
   <details><summary>Answer</summary>With self-loops the degrees are $\tilde d_1=\tilde d_3=2$, $\tilde d_2=3$. Node 2 sees itself and both neighbours: $\hat A_{22}=1/3$, $\hat A_{21}=\hat A_{23}=1/\sqrt{2\cdot3}=1/\sqrt6$. Output $=\mathrm{ReLU}(\tfrac13\cdot0+\tfrac1{\sqrt6}\cdot1+\tfrac1{\sqrt6}\cdot1)=2/\sqrt6\approx0.816$.</details>
2. Why is **sum** pooling used to turn atom vectors into a molecule vector, rather than mean or max?
   <details><summary>Answer</summary>Sum is permutation-invariant (any atom order gives the same total) and, unlike mean, remembers *counts*: two hydroxyls give twice the signal of one. A mean makes a molecule look like its own dimer; a max ignores multiplicity. Xu et al. (2019) showed sum is the most expressive of the three.</details>
3. A colleague reports $R^2=0.75$ for a molecular GNN on a random split and $R^2=0.1$ on a scaffold split. Which number better predicts how the model will do on a brand-new chemical series, and why?
   <details><summary>Answer</summary>The scaffold split (0.1): it holds out whole scaffolds, so the test resembles the real task of predicting for chemistry unseen in training. The random split leaks near-identical analogues into both sets and overstates prospective performance.</details>
4. Why are pairwise Cα–Cα distances a rotation- and translation-invariant description of a structure, and what information do they *lose*?
   <details><summary>Answer</summary>Rotating or translating a rigid body does not change the distance between any two of its atoms, so the distance matrix is invariant. It loses the absolute position/orientation (usually irrelevant) and **chirality**: a structure and its mirror image have identical pairwise distances.</details>
5. What is over-smoothing, and name one symptom you could measure and one architectural fix.
   <details><summary>Answer</summary>Repeated neighbour-averaging drives all node representations towards the same vector, so deep GNNs lose discriminative power. Symptom: the (normalised) Dirichlet energy of the node features falls towards 0 / accuracy drops with depth. Fixes: residual connections, normalisation, attention, or fewer layers.</details>
6. In what precise sense is a transformer a graph neural network?
   <details><summary>Answer</summary>Self-attention is graph attention on the **complete graph**: every token attends to every other. A GNN on a sparse graph is a transformer whose attention is masked to the edges; the adjacency matrix is the attention mask.</details>
7. The EGNN updates positions as $x_i' = x_i + \frac1{N-1}\sum_{j}(x_i-x_j)\,\phi_x(m_{ij})$ with $m_{ij}$ a function of distances only. Show in one line why this is rotation-equivariant.
   <details><summary>Answer</summary>Under a rotation $R$, each relative vector becomes $R(x_i-x_j)$ while distances — and therefore $m_{ij}$ and $\phi_x$ — are unchanged, so $x_i' \to Rx_i + \frac1{N-1}\sum_j R(x_i-x_j)\phi_x(m_{ij}) = R x_i'$: the output rotates with the input.</details>
8. Code reading: `GCN.forward` does `H = A_hat @ lin(H)`. Why does replacing `A_hat` by the identity matrix turn the GCN into an ordinary MLP, and why is that a useful baseline?
   <details><summary>Answer</summary>With $\hat A=I$ no neighbour information is mixed in: each layer is just `ReLU(H W)`, a standard MLP on each node's own features. It is the natural ablation that isolates how much the *graph* adds over the *features* alone.</details>

## 🧠 Questions & discussion

1. A GNN predicts protein function on a STRING network and reaches high accuracy. How would you check whether it learned real functional signal rather than the degree bias of well-studied proteins (Section 9)? Design the control.
2. When would you prefer a graph attention network over a plain GCN for a biological network, and when might the extra parameters of attention hurt? (Think about homophily, graph size and the number of labels.)
3. Weight tying across positions (CNN, Module 22), across nodes (GNN) and across orientations (EGNN) all encode a symmetry by sharing parameters. For a new problem — say predicting binding from a protein–ligand complex — how would you decide which symmetries to build in and which to leave to data augmentation?
4. Message passing is bounded by the 1-WL test. For molecular property prediction, is this limitation likely to matter in practice, given that real atoms carry rich features (element, charge, ring membership)? Argue both sides.
5. A model that moves atoms (an equivariant structure model) could in principle propose new protein or molecule structures. What are the dual-use and safety considerations of releasing such a model, and how do they differ from releasing a property *predictor* (discussed further in Modules 39 and 43)?

## 🚀 Bonus — going further

- Implement multi-head graph attention (several `GraphAttentionLayer`s in parallel, concatenate their outputs) and compare with the single head on the heterophilous (low-$h$) networks of Section 2.5.
- Add **Laplacian positional encodings**: compute the first few non-trivial eigenvectors of the graph Laplacian (Section 1) and concatenate them to the node features before the GCN. Do they help the hexagon-vs-triangles case of Section 5, and node classification?
- Turn the molecular MPNN into a **link-prediction** model for drug–target pairs: embed drugs and proteins with two encoders and score a pair by the dot product of their embeddings (the idea behind Decagon and many knowledge-graph models).
- Give the EGNN layer edge features (e.g. 'these two residues are consecutive in the chain') and train it to **denoise** a perturbed helix bundle back to its true coordinates — a miniature of a diffusion structure model ([Module 27](27_diffusion_flow_matching.ipynb)).
- Try a production library on real data (outside this course's dependencies): PyTorch Geometric on a small MoleculeNet task, or Chemprop on a ChEMBL assay, and compare its scaffold-split performance with the fingerprint baseline — the honest comparison of [Module 40](40_molecular_ml_drug_discovery.ipynb).

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 28 · Graph neural networks and geometric deep learning
Biological question: learn from relational/3-D biology (interaction networks, molecules, structures) while
                     respecting permutation and rotation/translation symmetry
Data (simulated): simulate_ppi_network (SBM, hidden modules); simulate_molecules (hidden SAR rules + oracle);
                  simulate_helix_bundle (exact 3-D coords of 3/4/5-helix bundles)
Method: GCN / label propagation / generic message passing; sum-pooling MPNN; graph attention; invariant distance
        features; E(n)-equivariant layer (EGNN)
Key equation: H' = σ(Â H W);  m_v = ⊕_u φ_msg(h_u,h_v,e_uv), h_v' = φ_upd(h_v, m_v);  EGNN x-update along x_i − x_j
Key parameters: n_layers (over-smoothing), homophily h, readout (sum/mean), split (random vs scaffold), d_hidden
Ground-truth check result: node classification GCN 0.97 vs MLP 0.69 / label spreading 0.72; molecule MPNN
                           R² ≈ 0.63 random, ≈ 0.2 scaffold (fingerprint models ≈ 0 or below; ceiling ≈ 0.9);
                           EGNN symmetry error ~1e-16; EGNN rotated-test accuracy 0.89 with no augmentation
What changed when I modified … (homophily / n_layers / split / mean vs sum pooling): …
One failure mode: the GCN hurts below homophily ≈ 1/2; raw-coordinate MLP fails on rotated structures; MPNN fails
                  across scaffolds
Connection to research: Kipf & Welling 2017; Gilmer et al. 2017; Xu et al. 2019; Satorras et al. 2021;
                        Stokes et al. 2020 (halicin); Jumper et al. 2021 (AlphaFold)
```

## 📚 References

- Kipf T. N., Welling M. (2017) Semi-supervised classification with graph convolutional networks. *ICLR*.
- Hamilton W. L., Ying R., Leskovec J. (2017) Inductive representation learning on large graphs. *NeurIPS* (GraphSAGE).
- Veličković P., Cucurull G., Casanova A., Romero A., Liò P., Bengio Y. (2018) Graph attention networks. *ICLR* (additive attention scoring).
- Vaswani A. *et al.* (2017) Attention is all you need. *NeurIPS* (scaled dot-product attention; the variant implemented in Section 6).
- Gilmer J., Schoenholz S. S., Riley P. F., Vinyals O., Dahl G. E. (2017) Neural message passing for quantum chemistry. *ICML*.
- Xu K., Hu W., Leskovec J., Jegelka S. (2019) How powerful are graph neural networks? *ICLR* (GIN).
- Weisfeiler B., Leman A. (1968) The reduction of a graph to canonical form and the algebra which appears therein. *Nauchno-Technicheskaya Informatsiya* (as cited in Xu et al. 2019).
- Morris C., Ritzert M., Fey M., Hamilton W. L., Lenssen J. E., Rattan G., Grohe M. (2019) Weisfeiler and Leman go neural: higher-order graph neural networks. *AAAI*.
- Li Q., Han Z., Wu X.-M. (2018) Deeper insights into graph convolutional networks for semi-supervised learning. *AAAI* (over-smoothing).
- Wu F., Souza A., Zhang T., Fifty C., Yu T., Weinberger K. (2019) Simplifying graph convolutional networks. *ICML* (SGC).
- Zhu X., Ghahramani Z. (2002) Learning from labeled and unlabeled data with label propagation. *CMU Tech. Report CMU-CALD-02-107* (label propagation, clamped labels).
- Zhou D., Bousquet O., Lal T. N., Weston J., Schölkopf B. (2004) Learning with local and global consistency. *NeurIPS* (label spreading, the F ← αSF + (1−α)Y₀ form implemented here).
- Holland P. W., Laskey K. B., Leinhardt S. (1983) Stochastic blockmodels: first steps. *Social Networks* 5:109–137.
- Bronstein M. M., Bruna J., Cohen T., Veličković P. (2021) Geometric deep learning: grids, groups, graphs, geodesics, and gauges. *arXiv:2104.13478*.
- Schütt K. T., Kindermans P.-J., Sauceda H. E., Chmiela S., Tkatchenko A., Müller K.-R. (2017) SchNet: a continuous-filter convolutional neural network for modeling quantum interactions. *NeurIPS*.
- Satorras V. G., Hoogeboom E., Welling M. (2021) E(n) equivariant graph neural networks. *ICML*.
- Dwivedi V. P., Joshi C. K., Luu A. T., Laurent T., Bengio Y., Bresson X. (2023) Benchmarking graph neural networks. *JMLR* 24(43):1–48 (Laplacian positional encodings).
- Stokes J. M. *et al.* (2020) A deep learning approach to antibiotic discovery. *Cell* 180:688–702 (halicin).
- Zitnik M., Agrawal M., Leskovec J. (2018) Modeling polypharmacy side effects with graph convolutional networks. *Bioinformatics* 34:i457–i466 (Decagon).
- Gligorijević V. *et al.* (2021) Structure-based protein function prediction using graph convolutional networks. *Nat Commun* 12:3168 (DeepFRI).
- Jumper J. *et al.* (2021) Highly accurate protein structure prediction with AlphaFold. *Nature* 596:583–589.

## ➡️ Next

A GNN or an MPNN gives a prediction; it does not tell you **which molecule to make next**, nor how confident it is.
**[Module 29 · Gaussian processes, Bayesian optimisation and active learning](29_gaussian_processes_bayesian_optimization.ipynb)**
adds calibrated uncertainty and a principled way to choose the next experiment — closing the design–build–test loop
that drug discovery ([Module 40](40_molecular_ml_drug_discovery.ipynb)) and protein design
([Module 39](39_generative_design.ipynb)) run on.

In [ ]:
# ── Total run time ──
print(f"⏱ Notebook compute: {time.time() - T_START:.0f} s on {torch.get_num_threads()} CPU thread(s).")